# 🏆 LB 2448 | Single-File Kaggriculture Agent — Evolved Tape + 7 Reactive Layers

A single-file agent. Public LB 2448 (submission 56112400, 2026-09-09).

**What it does**
- Replays a fixed 719-step schedule (a tape) for the farm and the market.
- The tape started from yhay81's shop-router route 0 and was evolved by a (1+λ) search against replays of live opponents (~150 of 719 steps changed).
- Seven reactive layers repair the replay each step: hand_align, weed_repair, sell_lead, front_run, room_guard, clamp_sells, terminal_liquidation.

**How to use**
- Run the notebook. It writes `main.py` and packs `submission.tar.gz`. Submit the archive.

**Credits / license**
- Chassis: ahmedberatozer (notebook07b5f4563e), thomastschinkel, tetsutani. Route data: yhay81 shop-router-0909. All Apache-2.0; this notebook is Apache-2.0 too.




In [ ]:
%%writefile main.py
# Kaggriculture agent: replays a fixed 719-step schedule inside a reactive chassis. Apache-2.0. Credits: thomastschinkel, yhay81, tetsutani (chassis/tapes).
from __future__ import annotations
import copy
PRODUCTS = ('WHEAT', 'CARROT', 'TOMATO', 'STRAWBERRY', 'MELON', 'EGG', 'MILK', 'WOOL', 'FERTILIZER')
SEED_PRICE = {'WHEAT': 10, 'CARROT': 20, 'TOMATO': 50, 'STRAWBERRY': 100, 'MELON': 80}
ANIMAL_COST = {'GOOSE': 300, 'COW': 400, 'SHEEP': 500}
ANIMAL_STRUCTURE = {'GOOSE': 'COOP', 'COW': 'PASTURE', 'SHEEP': 'PASTURE'}
LAND_PRICES = (1000, 2000, 4000)
MOVES = {'NORTH': (0, -1), 'SOUTH': (0, 1), 'EAST': (1, 0), 'WEST': (-1, 0)}
FRONT_RUN_ITEMS = ('MILK', 'WOOL', 'STRAWBERRY', 'MELON')
LAST_ACT_STEP = 718
PASS_ACTION = {'farmer': ['PASS'], 'hands': [], 'market': []}
DEFAULT_SETTINGS = {'hand_align': True, 'weed_repair': True, 'sell_lead': True, 'front_run': True, 'budget_guard': True, 'room_guard': True, 'clamp_sells': True, 'dead_stock': True, 'terminal_liquidation': True, 'block_turns': 72, 'shed_capacity': 100, 'board_size': 10, 'max_orders': 10, 'turns_per_day': 24, 'min_sell_price': 2}

def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, 'get', None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)

def _int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

def _fib(n):
    a, b = (1, 1)
    for _ in range(n):
        a, b = (b, a + b)
    return a

def _step_of(observation):
    raw = _get(observation, 'step')
    if raw is not None:
        return _int(raw)
    return _int(_get(observation, 'day', 0)) * 24 + _int(_get(observation, 'hour', 0))

def _shed_adjacent(pos, board):
    if not isinstance(pos, (list, tuple)) or len(pos) < 2:
        return False
    half = board // 2
    return pos[0] in (half - 1, half) and pos[1] in (half - 1, half)

def _tile_at(tiles, pos):
    try:
        x, y = (int(pos[0]), int(pos[1]))
        return tiles[y][x]
    except (TypeError, ValueError, IndexError):
        return 'LOCKED'

def _is_noop(act, tile, inv, seeds, pos, board):
    if not act:
        return True
    op = act[0]
    x, y = (pos[0], pos[1])
    if op in MOVES:
        dx, dy = MOVES[op]
        return not (0 <= x + dx < board and 0 <= y + dy < board)
    if op == 'PASS':
        return True
    adjacent = _shed_adjacent(pos, board)
    if op == 'DROP':
        return not adjacent or not inv
    if op == 'PICKUP':
        return not adjacent
    if op == 'PLACE':
        item = act[1] if len(act) > 1 else None
        if item in ANIMAL_STRUCTURE and isinstance(tile, dict) and (_get(tile, 'kind') == ANIMAL_STRUCTURE[item]) and (_get(tile, 'animal') is None):
            return _int(_get(inv, item, 0)) <= 0
        return not adjacent or _int(_get(inv, item, 0)) <= 0
    if tile == 'LOCKED':
        return True
    is_dict = isinstance(tile, dict)
    kind = _get(tile, 'kind') if is_dict else None
    animal = is_dict and _get(tile, 'animal') is not None
    if op == 'PLANT':
        return tile is not None or _int(_get(seeds, act[1] if len(act) > 1 else None, 0)) <= 0
    if op == 'WATER':
        return kind != 'PLANT' or bool(_get(tile, 'watered_today'))
    if op == 'HARVEST':
        return not is_dict or _int(_get(tile, 'yield_units', 0)) <= 0
    if op == 'FERTILIZE':
        return kind != 'PLANT' or _int(_get(inv, 'FERTILIZER', 0)) <= 0
    if op == 'DIG':
        return tile is None or animal
    if op in ('BUILD_COOP', 'BUILD_PASTURE'):
        return tile is not None
    if op == 'FEED':
        return not animal or bool(_get(tile, 'fed_today')) or _int(_get(inv, 'WHEAT', 0)) <= 0
    if op == 'COLLECT_FERTILIZER':
        return not animal or not _get(tile, 'fertilizer_available')
    if op == 'CARE':
        return not animal or bool(_get(tile, 'cared_today'))
    return True

class _View:

    def __init__(self, observation, player, cfg):
        farms = list(_get(observation, 'farms', []) or [])
        self.farm = farms[player] if player < len(farms) else {}
        self.rival = farms[1 - player] if len(farms) >= 2 and 1 - player < len(farms) else {}
        private = _get(observation, 'private', {}) or {}
        self.shed = {k: max(0, _int(v)) for k, v in dict(_get(private, 'shed', {}) or {}).items()}
        self.seeds = dict(_get(private, 'seeds', {}) or {})
        self.invs = [dict(i or {}) for i in _get(private, 'inventories', []) or []]
        market = _get(observation, 'market', {}) or {}
        self.prices = {k: _int(v) for k, v in dict(_get(market, 'prices', {}) or {}).items()}
        self.money = float(_get(self.farm, 'money', 0.0) or 0.0)
        self.tiles = _get(self.farm, 'tiles', []) or []
        self.board = len(self.tiles) or cfg['board_size']
        self.positions = [_get(self.farm, 'farmer', None)] + [list(p) for p in _get(self.farm, 'hands', []) or []]
        self.hires_today = _int(_get(self.farm, 'hires_today', 0))
        self.quadrants = len(list(_get(self.farm, 'unlocked_quadrants', []) or []))

    def inv(self, idx):
        return self.invs[idx] if idx < len(self.invs) else {}

    def in_hands(self, item):
        return sum((max(0, _int(_get(inv, item, 0))) for inv in self.invs))

class Chassis:

    def __init__(self, routes, router=None, settings=None, opponent_plan=None):
        self.routes = {rid: list(tape) for rid, tape in routes.items()}
        self.router = router or (lambda observation, step, state: next(iter(self.routes)))
        self.cfg = dict(DEFAULT_SETTINGS)
        self.cfg.update(settings or {})
        self.opponent_plan = opponent_plan
        self.players = {}
        self.diagnostics = {'layer_fallbacks': 0, 'entry_fallbacks': 0}
        self._future_sells = {}

    def _state(self, player, step):
        st = self.players.get(player)
        if st is None or step == 0 or step <= st['last_step']:
            st = {'last_step': -1, 'route': None, 'router_state': {}, 'pending': {}, 'sell_state': {'due_step': -1, 'suppress': {}}}
            self.players[player] = st
        st['last_step'] = step
        return st

    def _route_action(self, route, step):
        tape = self.routes[route]
        if 0 <= step < len(tape) and isinstance(tape[step], dict):
            return copy.deepcopy(tape[step])
        return copy.deepcopy(PASS_ACTION)

    def future_sells(self, route, item, step):
        table = self._future_sells.get(route)
        if table is None:
            tape = self.routes[route]
            n = len(tape)
            table = {p: [0] * (n + 1) for p in PRODUCTS}
            for t in range(n - 1, -1, -1):
                for p in PRODUCTS:
                    table[p][t] = table[p][t + 1]
                for o in tape[t].get('market') or [] if isinstance(tape[t], dict) else []:
                    if o and o[0] == 'SELL' and (len(o) >= 3) and (o[1] in table):
                        table[o[1]][t] += max(0, _int(o[2]))
            self._future_sells[route] = table
        col = table.get(item)
        return col[step] if col and 0 <= step < len(col) else 0

    def act(self, observation, configuration=None):
        if len(_get(observation, 'farms', []) or []) < 2:
            raise ValueError('incomplete observation')
        step = _step_of(observation)
        player = _int(_get(observation, 'player', 0))
        st = self._state(player, step)
        cfg = self.cfg
        view = _View(observation, player, cfg)
        route = self.router(observation, step, st['router_state'])
        if route not in self.routes:
            route = st['route'] if st['route'] in self.routes else next(iter(self.routes))
        st['route'] = route
        action = self._route_action(route, step)
        raw = copy.deepcopy(action)
        try:
            if cfg['hand_align']:
                self._hand_align(action, view)
            if cfg['weed_repair']:
                self._weed_repair(action, view, st, route, step)
            if cfg['sell_lead'] or cfg['front_run']:
                self._apply_suppression(action, st['sell_state'], step)
            projected = self._projected_shed(action, view)
            lead_available = dict(projected)
            next_sup = {'due_step': -1, 'suppress': {}}
            if cfg['sell_lead']:
                self._sell_lead(action, view, lead_available, route, step, next_sup)
            if cfg['front_run'] and self.opponent_plan:
                self._front_run(action, view, lead_available, route, step, next_sup)
            st['sell_state'] = next_sup
            if cfg['budget_guard']:
                self._budget_guard(action, view, route, step)
            if cfg['room_guard']:
                self._room_guard(action, view, route, step)
            if cfg['clamp_sells']:
                self._clamp_sells(action, projected)
            if cfg['dead_stock']:
                self._dead_stock(action, view, projected, route, step)
            if cfg['terminal_liquidation']:
                self._terminal_liquidation(action, projected, step)
            action['market'] = action['market'][:cfg['max_orders']]
            return action
        except Exception:
            self.diagnostics['layer_fallbacks'] += 1
            return raw

    def _hand_align(self, action, view):
        expected = max(0, len(view.positions) - 1)
        hands = list(action.get('hands') or [])
        hands.extend([['PASS'] for _ in range(max(0, expected - len(hands)))])
        action['hands'] = hands[:expected]

    def _weed_repair(self, action, view, st, route, step):
        units = [action.get('farmer') or ['PASS']] + list(action.get('hands') or [])
        pending = st['pending']
        tape = self.routes[route]
        nxt = tape[step + 1] if step + 1 < len(tape) and isinstance(tape[step + 1], dict) else {}
        next_units = [nxt.get('farmer') or ['PASS']] + list(nxt.get('hands') or [])
        for i in range(min(len(units), len(view.positions))):
            pos = view.positions[i]
            if not isinstance(pos, (list, tuple)):
                continue
            pos = (int(pos[0]), int(pos[1]))
            tile = _tile_at(view.tiles, pos)
            act = list(units[i])
            queue = pending.get(i)
            if queue and queue[0][0] != pos:
                pending.pop(i, None)
                queue = None
            is_weed = isinstance(tile, dict) and _get(tile, 'kind') == 'WEED'
            noop = _is_noop(act, tile, view.inv(i), view.seeds, pos, view.board)
            next_op = next_units[i][0] if i < len(next_units) and next_units[i] else 'PASS'
            if act and act[0] in ('PLANT', 'BUILD_COOP', 'BUILD_PASTURE') and is_weed:
                pending.setdefault(i, []).append((pos, act))
                act = ['DIG']
            elif queue and noop:
                _, replay = queue[0]
                if replay[0] == 'PLANT' and next_op in MOVES:
                    pending.pop(i, None)
                else:
                    queue.pop(0)
                    if act and act[0] != 'PASS' and (act[0] not in MOVES):
                        queue.append((pos, act))
                    act = replay
                    if not queue:
                        pending.pop(i, None)
            elif is_weed and noop:
                act = ['DIG']
            units[i] = act
        action['farmer'] = units[0]
        action['hands'] = units[1:]

    def _projected_shed(self, action, view):
        cap = self.cfg['shed_capacity']
        proj = {p: view.shed.get(p, 0) for p in PRODUCTS}
        for k, v in view.shed.items():
            proj.setdefault(k, v)
        total = sum(proj.values())
        units = [action.get('farmer') or ['PASS']] + list(action.get('hands') or [])
        for i in range(min(len(units), len(view.positions))):
            if not _shed_adjacent(view.positions[i], view.board):
                continue
            act = units[i]
            op = act[0] if act else 'PASS'
            inv = view.inv(i)
            if op == 'PICKUP' and len(act) >= 2 and (act[1] in proj):
                qty = min(proj[act[1]], max(0, _int(act[2]) if len(act) >= 3 else 1))
                proj[act[1]] -= qty
                total -= qty
            elif op == 'DROP':
                for item, held in inv.items():
                    take = min(max(0, _int(held)), max(0, cap - total))
                    if take > 0:
                        proj[item] = proj.get(item, 0) + take
                        total += take
            elif op == 'PLACE' and len(act) >= 2 and (act[1] not in ANIMAL_STRUCTURE):
                item = act[1]
                take = min(max(0, _int(act[2]) if len(act) >= 3 else 1), max(0, _int(_get(inv, item, 0))), max(0, cap - total))
                if take > 0:
                    proj[item] = proj.get(item, 0) + take
                    total += take
        return proj

    @staticmethod
    def _apply_suppression(action, sell_state, step):
        if sell_state.get('due_step') != step:
            return
        remaining = dict(sell_state.get('suppress', {}))
        kept = []
        for order in action.get('market') or []:
            order = list(order)
            if order and order[0] == 'SELL' and (len(order) >= 3) and (remaining.get(order[1], 0) > 0):
                removed = min(max(0, _int(order[2])), remaining[order[1]])
                order[2] = _int(order[2]) - removed
                remaining[order[1]] -= removed
            kept.append(order)
        action['market'] = kept

    @staticmethod
    def _add_sell(action, item, qty, max_orders, merge=True):
        market = action.setdefault('market', [])
        if merge:
            for order in market:
                if order and order[0] == 'SELL' and (order[1] == item):
                    order[2] = _int(order[2]) + qty
                    return True
        if len(market) >= max_orders:
            return False
        market.append(['SELL', item, qty])
        return True

    def _sell_lead(self, action, view, projected, route, step, next_sup):
        cfg = self.cfg
        nxt = step + 1
        unlock_period = 3 * cfg['turns_per_day']
        if nxt > LAST_ACT_STEP or nxt % unlock_period == 0 or step % 4 == 0:
            return
        tape = self.routes[route]
        future = tape[nxt] if nxt < len(tape) and isinstance(tape[nxt], dict) else {}
        planned = {}
        for o in future.get('market') or []:
            if o and o[0] == 'SELL' and (len(o) >= 3) and (o[1] in PRODUCTS):
                planned[o[1]] = planned.get(o[1], 0) + max(0, _int(o[2]))
        already = {o[1] for o in action.get('market') or [] if o and o[0] == 'SELL' and (len(o) > 1)}
        for item in PRODUCTS:
            if item in ('WHEAT', 'FERTILIZER') or planned.get(item, 0) <= 0 or item in already:
                continue
            qty = min(projected.get(item, 0), planned[item])
            if qty <= 0 or view.prices.get(item, 0) < cfg['min_sell_price']:
                continue
            if not self._add_sell(action, item, qty, cfg['max_orders'], merge=False):
                break
            projected[item] -= qty
            next_sup['suppress'][item] = next_sup['suppress'].get(item, 0) + qty
        if next_sup['suppress']:
            next_sup['due_step'] = nxt

    def _front_run(self, action, view, projected, route, step, next_sup):
        cfg = self.cfg
        nxt = step + 1
        plan = self.opponent_plan
        if nxt > LAST_ACT_STEP or nxt >= len(plan) or (not isinstance(plan[nxt], dict)):
            return
        already = {o[1] for o in action.get('market') or [] if o and o[0] == 'SELL' and (len(o) > 1)}
        for o in plan[nxt].get('market') or []:
            if not (o and o[0] == 'SELL' and (len(o) >= 3) and (o[1] in FRONT_RUN_ITEMS)):
                continue
            item = o[1]
            if item in already or view.prices.get(item, 0) < cfg['min_sell_price']:
                continue
            own_next = sum((max(0, _int(x[2])) for x in self.routes[route][nxt].get('market', []) if len(x) >= 3 and x[0] == 'SELL' and (x[1] == item)))
            qty = min(projected.get(item, 0), max(0, _int(o[2])), own_next)
            if qty <= 0:
                continue
            if not self._add_sell(action, item, qty, cfg['max_orders'], merge=False):
                break
            projected[item] -= qty
            already.add(item)
            next_sup['suppress'][item] = next_sup['suppress'].get(item, 0) + qty
        if next_sup['suppress']:
            next_sup['due_step'] = nxt

    def _block_requirements(self, view, route, start, end):
        tape = self.routes[route]
        budget = 0.0
        seed_bal, item_bal = ({}, {})
        seed_need, item_need = ({}, {})
        hires_by_day = {}
        quadrants = view.quadrants
        for t in range(start, min(end, len(tape))):
            a = tape[t] if isinstance(tape[t], dict) else {}
            for u in [a.get('farmer') or ['PASS']] + list(a.get('hands') or []):
                if not u:
                    continue
                op = u[0]
                arg = u[1] if len(u) > 1 else None
                qty = max(1, _int(u[2]) if len(u) > 2 else 1)
                if op == 'PLANT' and arg in SEED_PRICE:
                    seed_bal[arg] = seed_bal.get(arg, 0) - 1
                    seed_need[arg] = max(seed_need.get(arg, 0), -seed_bal[arg])
                elif op == 'FEED':
                    item_bal['WHEAT'] = item_bal.get('WHEAT', 0) - 1
                    item_need['WHEAT'] = max(item_need.get('WHEAT', 0), -item_bal['WHEAT'])
                elif op == 'FERTILIZE':
                    item_bal['FERTILIZER'] = item_bal.get('FERTILIZER', 0) - 1
                    item_need['FERTILIZER'] = max(item_need.get('FERTILIZER', 0), -item_bal['FERTILIZER'])
                elif op == 'PLACE' and arg is not None:
                    item_bal[arg] = item_bal.get(arg, 0) - qty
                    item_need[arg] = max(item_need.get(arg, 0), -item_bal[arg])
            for o in a.get('market') or []:
                if not o:
                    continue
                op = o[0]
                item = o[1] if len(o) > 1 else None
                qty = max(1, _int(o[2]) if len(o) > 2 else 1)
                if op == 'HIRE':
                    day = (t - start) // self.cfg['turns_per_day']
                    hires_by_day[day] = hires_by_day.get(day, 0) + 1
                elif op == 'BUY_LAND':
                    extra = quadrants - 1
                    if 0 <= extra < len(LAND_PRICES):
                        budget += LAND_PRICES[extra]
                        quadrants += 1
                elif op == 'BUY_SEED' and item in SEED_PRICE:
                    budget += SEED_PRICE[item] * qty
                    seed_bal[item] = seed_bal.get(item, 0) + qty
                elif op == 'BUY_PRODUCT' and item in ('WHEAT', 'FERTILIZER'):
                    budget += view.prices.get(item, 0) * qty
                    item_bal[item] = item_bal.get(item, 0) + qty
                elif op == 'BUY_ANIMAL' and item in ANIMAL_COST:
                    budget += ANIMAL_COST[item] * qty
                    item_bal[item] = item_bal.get(item, 0) + qty
        for day, n in hires_by_day.items():
            first = view.hires_today if day == 0 else 0
            for k in range(n):
                budget += _fib(first + k)
        return (budget, item_need)

    def _budget_guard(self, action, view, route, step):
        cfg = self.cfg
        block = cfg['block_turns']
        if block <= 0 or step % block != 0:
            return
        budget, item_need = self._block_requirements(view, route, step, step + block)
        market = action.setdefault('market', [])
        existing = {}
        for o in market:
            if o and o[0] == 'SELL' and (len(o) >= 3):
                existing[o[1]] = existing.get(o[1], 0) + max(0, _int(o[2]))
        cash = view.money
        for item in PRODUCTS:
            planned = max(existing.get(item, 0), self.future_sells(route, item, step) - self.future_sells(route, item, step + block))
            cash += min(view.shed.get(item, 0), planned) * view.prices.get(item, 0)
        shortfall = budget - cash
        if shortfall <= 0:
            return
        candidates = []
        for item in PRODUCTS:
            price = view.prices.get(item, 0)
            if price < cfg['min_sell_price']:
                continue
            protected = max(0, item_need.get(item, 0) - view.in_hands(item))
            avail = view.shed.get(item, 0) - protected - existing.get(item, 0)
            if avail > 0:
                candidates.append((-price, item, avail, price))
        candidates.sort()
        added = False
        for _, item, avail, price in candidates:
            if shortfall <= 0:
                break
            qty = min(avail, -(-int(shortfall) // price))
            if self._add_sell(action, item, qty, cfg['max_orders']):
                shortfall -= qty * price
                added = True
        if added:
            sells = [o for o in market if o and o[0] == 'SELL']
            others = [o for o in market if not (o and o[0] == 'SELL')]
            action['market'] = sells + others

    def _room_guard(self, action, view, route, step):
        cfg = self.cfg
        if step % cfg['turns_per_day'] != cfg['turns_per_day'] - 1:
            return
        cap = cfg['shed_capacity']
        units = [action.get('farmer') or ['PASS']] + list(action.get('hands') or [])
        carried = sum((max(0, _int(n)) for inv in view.invs for n in inv.values()))
        produced = consumed = 0
        for i in range(min(len(units), len(view.positions))):
            tile = _tile_at(view.tiles, view.positions[i])
            a = units[i]
            if not a:
                continue
            op = a[0]
            if op == 'HARVEST' and isinstance(tile, dict):
                produced += max(0, _int(_get(tile, 'yield_units', 0)))
            elif op == 'COLLECT_FERTILIZER' and isinstance(tile, dict) and _get(tile, 'fertilizer_available'):
                produced += 1
            elif op in ('FEED', 'FERTILIZE'):
                consumed += 1
            elif op == 'PLACE' and len(a) > 1 and (a[1] in ANIMAL_STRUCTURE):
                consumed += 1
        market = action.setdefault('market', [])
        planned_sells, planned_buys = ({}, 0)
        for o in market:
            if not o:
                continue
            if o[0] == 'SELL' and len(o) >= 3:
                planned_sells[o[1]] = planned_sells.get(o[1], 0) + max(0, _int(o[2]))
            elif o[0] in ('BUY_PRODUCT', 'BUY_ANIMAL') and len(o) >= 3:
                planned_buys += max(0, _int(o[2]))
        shed_total = sum(view.shed.values())
        fillable = sum((min(view.shed.get(it, 0), n) for it, n in planned_sells.items()))
        needed = shed_total + carried + produced - consumed + planned_buys - fillable - (cap - 1)
        if needed <= 0:
            return
        priority = sorted(PRODUCTS, key=lambda it: (self.future_sells(route, it, step + 1) > 0, -view.prices.get(it, 0), it))
        for item in priority:
            avail = max(0, view.shed.get(item, 0) - planned_sells.get(item, 0))
            qty = min(needed, avail)
            if qty <= 0 or view.prices.get(item, 0) < 1:
                continue
            if not self._add_sell(action, item, qty, cfg['max_orders']):
                continue
            planned_sells[item] = planned_sells.get(item, 0) + qty
            needed -= qty
            if needed <= 0:
                break

    @staticmethod
    def _clamp_sells(action, projected):
        avail = dict(projected)
        kept = []
        for o in action.get('market') or []:
            if o and o[0] == 'SELL' and (len(o) >= 3):
                have = avail.get(o[1], 0)
                n = min(_int(o[2]), have)
                n = max(0, n)
                avail[o[1]] = have - n
                kept.append(['SELL', o[1], n])
            else:
                kept.append(o)
                if o and o[0] in ('BUY_PRODUCT', 'BUY_ANIMAL') and (len(o) >= 3):
                    avail[o[1]] = avail.get(o[1], 0) + max(0, _int(o[2]))
        action['market'] = kept

    def _dead_stock(self, action, view, projected, route, step):
        planned = {}
        for o in action.get('market') or []:
            if o and o[0] == 'SELL' and (len(o) >= 3):
                planned[o[1]] = planned.get(o[1], 0) + _int(o[2])
        day = step // self.cfg['turns_per_day']
        extra = []
        for item in PRODUCTS:
            have = projected.get(item, 0) - planned.get(item, 0)
            if have <= 0:
                continue
            surplus = have if day >= 29 else have - self.future_sells(route, item, step + 1)
            if surplus > 0 and view.prices.get(item, 0) > 1:
                extra.append(['SELL', item, surplus])
        extra.sort(key=lambda o: -view.prices.get(o[1], 0) * o[2])
        action['market'] = (action.get('market') or []) + extra

    def _terminal_liquidation(self, action, projected, step):
        if step < LAST_ACT_STEP:
            return
        action['market'] = [['SELL', item, qty] for item, qty in projected.items() if qty > 0 and item in PRODUCTS][:self.cfg['max_orders']]

def make_agent(routes, router=None, opponent_plan=None, **settings):
    chassis = Chassis(routes, router, settings, opponent_plan)

    def agent(observation, configuration=None):
        try:
            return chassis.act(observation, configuration)
        except Exception:
            chassis.diagnostics['entry_fallbacks'] += 1
            try:
                step = _step_of(observation)
                player = _int(_get(observation, 'player', 0))
                tape = chassis.routes.get(chassis.players.get(player, {}).get('route'), next(iter(chassis.routes.values())))
                if 0 <= step < len(tape):
                    return copy.deepcopy(tape[step])
            except Exception:
                pass
            try:
                farms = _get(observation, 'farms', []) or []
                hands = _get(farms[_int(_get(observation, 'player', 0))], 'hands', []) or []
                return {'farmer': ['PASS'], 'hands': [['PASS'] for _ in hands], 'market': []}
            except Exception:
                return copy.deepcopy(PASS_ACTION)
    agent.chassis = chassis
    return agent
import base64
import json
_TAPE = json.loads('[{"farmer":["PASS"],"hands":[],"market":[["BUY_PRODUCT","WHEAT",13],["SELL","WHEAT",10],["BUY_PRODUCT","WHEAT",13]]},{"farmer":["NORTH"],"hands":[],"market":[["SELL","WHEAT",10],["BUY_PRODUCT","WHEAT",5],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["BUY_ANIMAL","COW",2],["BUY_ANIMAL","SHEEP",2]]},{"farmer":["SOUTH"],"hands":[["PICKUP","COW"],["PASS"],["NORTH"],["NORTH"],["PICKUP","COW"]],"market":[]},{"farmer":["PICKUP","SHEEP"],"hands":[["NORTH"],["PASS"],["NORTH"],["NORTH"],["BUILD_PASTURE"]],"market":[]},{"farmer":["PICKUP","WHEAT"],"hands":[["BUILD_PASTURE"],["PASS"],["NORTH"],["NORTH"],["PLACE","COW"]],"market":[]},{"farmer":["NORTH"],"hands":[["PLACE","COW"],["PASS"],["NORTH"],["WEST"],["CARE"]],"market":[]},{"farmer":["WEST"],"hands":[["NORTH"],["PASS"],["NORTH"],["WEST"],["PICKUP","SHEEP"]],"market":[["BUY_SEED","MELON",2]]},{"farmer":["BUILD_PASTURE"],"hands":[["NORTH"],["PASS"],["PLANT","MELON"],["PLANT","MELON"],["PICKUP","WHEAT"]],"market":[["BUY_SEED","MELON",1]]},{"farmer":["PLACE","SHEEP"],"hands":[["PLANT","MELON"],["PASS"],["WATER"],["WATER"],["WEST"]],"market":[]},{"farmer":["FEED"],"hands":[["WATER"],["PASS"],["WEST"],["NORTH"],["BUILD_PASTURE"]],"market":[["BUY_SEED","MELON",1]]},{"farmer":["CARE"],"hands":[["WEST"],["PASS"],["PLANT","MELON"],["WEST"],["PLACE","SHEEP"]],"market":[["BUY_SEED","MELON",2]]},{"farmer":["WEST"],"hands":[["PLANT","MELON"],["PASS"],["WATER"],["PLANT","MELON"],["FEED"]],"market":[["BUY_SEED","MELON",1]]},{"farmer":["PLANT","MELON"],"hands":[["WATER"],["PASS"],["WEST"],["WATER"],["CARE"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["SOUTH"],["PASS"],["PLANT","WHEAT"],["WEST"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["WEST"],["PASS"],["WATER"],["PLANT","WHEAT"],["WEST"]],"market":[["BUY_SEED","MELON",3]]},{"farmer":["PLANT","MELON"],"hands":[["PLANT","MELON"],["PASS"],["WEST"],["WATER"],["PLANT","MELON"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WATER"],["PASS"],["PLANT","WHEAT"],["WEST"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["WEST"],["PASS"],["WATER"],["PLANT","WHEAT"],["WEST"]],"market":[["BUY_SEED","MELON",2]]},{"farmer":["PASS"],"hands":[["PLANT","MELON"],["PASS"],["PASS"],["WATER"],["PLANT","MELON"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["PASS"],["PASS"],["NORTH"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["WEST"],["PASS"],["PASS"],["PLANT","WHEAT"],["NORTH"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["PASS"],"hands":[["PLANT","WHEAT"],["PASS"],["PASS"],["WATER"],["PLANT","WHEAT"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["PASS"],["PASS"],["PASS"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"],["PASS"],["PASS"]],"market":[]},{"farmer":["PICKUP","WHEAT"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["NORTH"],"hands":[["WEST"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT"],["PICKUP","WHEAT"],["WEST"]],"market":[]},{"farmer":["FEED"],"hands":[["NORTH"],["FEED"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["FEED"],["CARE"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["BUILD_PASTURE"]],"market":[]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["PLACE","FERTILIZER"],["NORTH"]],"market":[["SELL","FERTILIZER",1],["BUY_PRODUCT","WHEAT",3]]},{"farmer":["EAST"],"hands":[["SOUTH"],["NORTH"],["WEST"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["PLACE","FERTILIZER"],"hands":[["PLACE","FERTILIZER"],["NORTH"],["WEST"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["PASS"],"hands":[["PASS"],["BUILD_PASTURE"],["WATER"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PASS"],["SOUTH"],["NORTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["SOUTH"],["WATER"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PASS"],["PICKUP","WHEAT"],["NORTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["WEST"],["EAST"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PASS"],["FEED"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["COLLECT_FERTILIZER"],["WEST"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PASS"],["CARE"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["NORTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["EAST"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["EAST"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"]],"market":[]},{"farmer":["PICKUP","WHEAT",4],"hands":[],"market":[["SELL","FERTILIZER",1],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["FEED"],"hands":[["PASS"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["CARE"],"hands":[["PASS"],["WEST"],["NORTH"],["NORTH"]],"market":[["SELL","WHEAT",2]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["WEST"],["NORTH"],["WEST"]],"market":[]},{"farmer":["NORTH"],"hands":[["PASS"],["WEST"],["NORTH"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["PASS"],["WEST"],["WEST"],["NORTH"]],"market":[]},{"farmer":["CARE"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["EAST"],["NORTH"],["WEST"]],"market":[]},{"farmer":["WEST"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["PASS"],["NORTH"],["WEST"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["PASS"],["WATER"],["WATER"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["NORTH"],["WEST"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["PASS"],["WATER"],["WEST"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["FEED"],"hands":[["PASS"],["EAST"],["WATER"],["PLANT","WHEAT"]],"market":[]},{"farmer":["CARE"],"hands":[["PASS"],["WATER"],["HARVEST"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["SOUTH"],["PLANT","WHEAT"],["NORTH"]],"market":[]},{"farmer":["EAST"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["PLACE","FERTILIZER",4],"hands":[["PASS"],["WEST"],["EAST"],["HARVEST"]],"market":[["SELL","FERTILIZER",4],["BUY_ANIMAL","COW",1],["BUY_SEED","WHEAT",1]]},{"farmer":["PICKUP","COW"],"hands":[["PASS"],["WEST"],["WATER"],["PLANT","WHEAT"]],"market":[]},{"farmer":["NORTH"],"hands":[["PASS"],["WATER"],["PASS"],["WATER"]],"market":[]},{"farmer":["NORTH"],"hands":[["PASS"],["NORTH"],["PASS"],["SOUTH"]],"market":[]},{"farmer":["PLACE","COW"],"hands":[["PASS"],["WATER"],["PASS"],["EAST"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"],["PASS"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["PICKUP","WHEAT",3],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["BUY_SEED","WHEAT",5]]},{"farmer":["WEST"],"hands":[["WEST"],["WEST"],["WEST"],["WEST"],["WEST"]],"market":[["SELL","WHEAT",2]]},{"farmer":["FEED"],"hands":[["WEST"],["WEST"],["NORTH"],["WEST"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["WEST"],["WEST"],["NORTH"],["WEST"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["NORTH"],["NORTH"],["NORTH"],["WEST"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["WATER"],["NORTH"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["FEED"],"hands":[["WEST"],["NORTH"],["WATER"],["WATER"],["NORTH"]],"market":[]},{"farmer":["CARE"],"hands":[["WATER"],["NORTH"],["NORTH"],["HARVEST"],["NORTH"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["SOUTH"],["WATER"],["WATER"],["PLANT","WHEAT"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["WATER"],["HARVEST"],["WEST"],["WATER"],["HARVEST"]],"market":[]},{"farmer":["NORTH"],"hands":[["WEST"],["PLANT","WHEAT"],["WATER"],["NORTH"],["PLANT","WHEAT"]],"market":[]},{"farmer":["FEED"],"hands":[["WATER"],["WATER"],["SOUTH"],["WATER"],["WATER"]],"market":[]},{"farmer":["CARE"],"hands":[["EAST"],["NORTH"],["WATER"],["HARVEST"],["SOUTH"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["EAST"],["WATER"],["SOUTH"],["PLANT","WHEAT"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["EAST"],["EAST"],["WATER"],["WATER"],["SOUTH"]],"market":[]},{"farmer":["SOUTH"],"hands":[["EAST"],["EAST"],["PASS"],["NORTH"],["WATER"]],"market":[]},{"farmer":["PLACE","FERTILIZER",3],"hands":[["NORTH"],["EAST"],["PASS"],["WATER"],["WEST"]],"market":[["SELL","FERTILIZER",3],["BUY_ANIMAL","COW",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["SOUTH"],["EAST"],["NORTH"],["WATER"]],"market":[]},{"farmer":["PLACE","FERTILIZER"],"hands":[["SOUTH"],["SOUTH"],["SOUTH"],["WATER"],["EAST"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["CARE"],"hands":[["CARE"],["SOUTH"],["SOUTH"],["PASS"],["EAST"]],"market":[]},{"farmer":["NORTH"],"hands":[["PASS"],["SOUTH"],["PICKUP","COW"],["PASS"],["EAST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["DROP"],["WEST"],["PASS"],["SOUTH"]],"market":[]},{"farmer":["CARE"],"hands":[["PASS"],["PASS"],["WEST"],["PASS"],["SOUTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PLACE","COW"],["PASS"],["DROP"]],"market":[]},{"farmer":["PICKUP","WHEAT",5],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["FEED"],"hands":[["NORTH"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["CARE"],"hands":[["WEST"],["NORTH"],["NORTH"],["NORTH"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WEST"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["CARE"],["NORTH"],["WEST"],["WEST"]],"market":[["SELL","WHEAT",2]]},{"farmer":["FEED"],"hands":[["SOUTH"],["NORTH"],["WEST"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["CARE"],["WEST"],["WATER"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["NORTH"],["WEST"],["NORTH"],["WEST"]],"market":[]},{"farmer":["WEST"],"hands":[["NORTH"],["WEST"],["EAST"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["EAST"],["WATER"],["WATER"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["HARVEST"],["SOUTH"],["PLANT","WHEAT"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["SOUTH"],["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["SOUTH"],["WATER"],["SOUTH"],["NORTH"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["FEED"],"hands":[["WEST"],["EAST"],["SOUTH"],["WATER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WEST"],["WATER"],["PASS"],["HARVEST"]],"market":[["BUY_PRODUCT","WHEAT",1],["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["SOUTH"],["PLACE","FERTILIZER"],["PLANT","WHEAT"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[["COLLECT_FERTILIZER"],["SOUTH"],["PASS"],["WATER"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["FEED"],"hands":[["PASS"],["SOUTH"],["PASS"],["SOUTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["SOUTH"],["PASS"],["SOUTH"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["EAST"],["PASS"],["SOUTH"]],"market":[]},{"farmer":["EAST"],"hands":[["PASS"],["EAST"],["PASS"],["WATER"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["EAST"],"hands":[["PASS"],["DROP"],["PASS"],["PASS"]],"market":[]},{"farmer":["PLACE","FERTILIZER",5],"hands":[["PASS"],["PASS"],["PASS"],["PASS"]],"market":[["SELL","FERTILIZER",4],["BUY_PRODUCT","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"],["PASS"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["EAST"],"hands":[["PICKUP","WHEAT",2],["PASS"],["NORTH"]],"market":[["HIRE"],[],["SELL","FERTILIZER",1]]},{"farmer":["PICKUP","WHEAT",2],"hands":[["FEED"],["WEST"],["PASS"],["NORTH"]],"market":[["SELL","WHEAT",2]]},{"farmer":["WEST"],"hands":[["CARE"],["PICKUP","WHEAT",2],["COLLECT_FERTILIZER"],["NORTH"]],"market":[]},{"farmer":["FEED"],"hands":[["WEST"],["NORTH"],["WEST"],["NORTH"]],"market":[]},{"farmer":["CARE"],"hands":[["WEST"],["FEED"],["WEST"],["NORTH"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["FEED"],["CARE"],["WEST"],["WATER"]],"market":[]},{"farmer":["WEST"],"hands":[["CARE"],["NORTH"],["WATER"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["EAST"],["FEED"],["WEST"],["PASS"]],"market":[]},{"farmer":["NORTH"],"hands":[["EAST"],["CARE"],["WATER"],["PASS"]],"market":[]},{"farmer":["WATER"],"hands":[["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["PASS"]],"market":[["BUY_SEED","STRAWBERRY",2]]},{"farmer":["NORTH"],"hands":[["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["PASS"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WATER"],"hands":[["EAST"],["NORTH"],["HARVEST"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["SOUTH"],["WATER"],["PLANT","STRAWBERRY"],["WEST"]],"market":[]},{"farmer":["WATER"],"hands":[["DROP"],["WEST"],["WATER"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["PASS"],["WATER"],["EAST"],["WEST"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["FEED"],"hands":[["PASS"],["WEST"],["WATER"],["WATER"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["CARE"],"hands":[["PASS"],["WATER"],["NORTH"],["HARVEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PASS"],["PASS"],["WATER"],["PLANT","STRAWBERRY"]],"market":[]},{"farmer":["EAST"],"hands":[["PASS"],["HARVEST"],["WEST"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["PASS"],["PLANT","STRAWBERRY"],["WATER"],["WEST"]],"market":[]},{"farmer":["DROP"],"hands":[["PASS"],["WATER"],["HARVEST"],["WATER"]],"market":[["SELL","FERTILIZER",4]]},{"farmer":["PASS"],"hands":[["PASS"],["WEST"],["PLANT","STRAWBERRY"],["NORTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["NORTH"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",2],["WEST"],["NORTH"],["WEST"],["PICKUP","WHEAT"],["WEST"],["NORTH"]],"market":[["SELL","FERTILIZER",1],[]]},{"farmer":["HARVEST"],"hands":[["WEST"],["PICKUP","WHEAT"],["PICKUP","WHEAT",2],["WEST"],["FEED"],["PICKUP","WHEAT"],["PICKUP","WHEAT"]],"market":[]},{"farmer":["EAST"],"hands":[["FEED"],["NORTH"],["WEST"],["WEST"],["CARE"],["NORTH"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["FEED"],["WEST"],["NORTH"],["WEST"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["DROP"],"hands":[["EAST"],["CARE"],["FEED"],["NORTH"],["CARE"],["FEED"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["EAST"],"hands":[["PLACE","WOOL",6],["NORTH"],["CARE"],["WATER"],["COLLECT_FERTILIZER"],["CARE"],["SOUTH"]],"market":[["SELL","WOOL",30],["SELL","FERTILIZER",1],["BUY_PRODUCT","WHEAT",2],["BUY_LAND"],["BUY_ANIMAL","COW",2],["BUY_PRODUCT","FERTILIZER",1]]},{"farmer":["EAST"],"hands":[["EAST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["NORTH"],["EAST"],["EAST"],["PICKUP","COW"]],"market":[["SELL","FERTILIZER",1],["BUY_PRODUCT","FERTILIZER",1]]},{"farmer":["EAST"],"hands":[["PICKUP","COW"],["SOUTH"],["EAST"],["PASS"],["EAST"],["PASS"],["EAST"]],"market":[["BUY_PRODUCT","WHEAT",2],["BUY_PRODUCT","WHEAT",3]]},{"farmer":["NORTH"],"hands":[["BUILD_PASTURE"],["SOUTH"],["EAST"],["PASS"],["DROP"],["BUILD_PASTURE"],["PLACE","FERTILIZER",2]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["DROP"],["PLACE","FERTILIZER"],["PASS"],["WEST"],["EAST"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PLACE","COW"],["NORTH"],["WEST"],["WATER"],["WEST"],["BUILD_PASTURE"],["BUILD_PASTURE"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["PASS"],"hands":[["FEED"],["NORTH"],["NORTH"],["NORTH"],["WEST"],["NORTH"],["PLACE","COW"]],"market":[["BUY_SEED","STRAWBERRY",2],["HIRE"]]},{"farmer":["PASS"],"hands":[["CARE"],["NORTH"],["FEED"],["WATER"],["WEST"],["PLANT","STRAWBERRY"],["FEED"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["PASS"],"hands":[["EAST"],["WATER"],["CARE"],["WEST"],["WATER"],["WATER"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["PLANT","STRAWBERRY"],"hands":[["BUILD_PASTURE"],["EAST"],["COLLECT_FERTILIZER"],["NORTH"],["NORTH"],["EAST"],["EAST"]],"market":[["BUY_SEED","STRAWBERRY",1],["BUY_SEED","WHEAT",2]]},{"farmer":["WATER"],"hands":[["EAST"],["PLANT","STRAWBERRY"],["NORTH"],["WATER"],["WATER"],["PLANT","WHEAT"],["BUILD_PASTURE"]],"market":[]},{"farmer":["EAST"],"hands":[["BUILD_PASTURE"],["WATER"],["WATER"],["HARVEST"],["NORTH"],["WATER"],["EAST"]],"market":[]},{"farmer":["PASS"],"hands":[["EAST"],["NORTH"],["NORTH"],["WEST"],["WATER"],["NORTH"],["NORTH"]],"market":[["BUY_SEED","STRAWBERRY",3],["BUY_SEED","WHEAT",1]]},{"farmer":["PLANT","STRAWBERRY"],"hands":[["PLANT","STRAWBERRY"],["PLANT","WHEAT"],["WATER"],["WATER"],["WEST"],["PLANT","WHEAT"],["PLANT","STRAWBERRY"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["WATER"],["NORTH"],["HARVEST"],["SOUTH"],["WATER"],["WATER"]],"market":[["SELL","WHEAT",7]]},{"farmer":["PASS"],"hands":[["EAST"],["EAST"],["WATER"],["SOUTH"],["SOUTH"],["EAST"],["EAST"]],"market":[["BUY_SEED","STRAWBERRY",2],["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PLANT","STRAWBERRY"],["PLANT","WHEAT"],["EAST"],["WATER"],["WATER"],["PLANT","WHEAT"],["PLANT","STRAWBERRY"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["WATER"],"hands":[["WATER"],["WATER"],["WATER"],["HARVEST"],["PASS"],["WATER"],["WATER"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["PICKUP","WHEAT"],"hands":[],"market":[["SELL","WOOL",3],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["BUY_SEED","WHEAT",4],["BUY_SEED","STRAWBERRY",4]]},{"farmer":["WEST"],"hands":[["WEST"],["NORTH"],["NORTH"],["PICKUP","WHEAT"],["WEST"],["NORTH"]],"market":[["HIRE"],["HIRE"],["BUY_ANIMAL","COW",1]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT"],["PICKUP","WHEAT"],["PICKUP","WHEAT"],["WEST"],["PICKUP","WHEAT"],["PICKUP","WHEAT"],["WEST"],["WEST"]],"market":[]},{"farmer":["FEED"],"hands":[["WEST"],["NORTH"],["FEED"],["NORTH"],["NORTH"],["FEED"],["WEST"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["FEED"],["NORTH"],["CARE"],["FEED"],["FEED"],["CARE"],["NORTH"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["FEED"],["COLLECT_FERTILIZER"],["CARE"],["CARE"],["COLLECT_FERTILIZER"],["NORTH"],["WEST"]],"market":[]},{"farmer":["EAST"],"hands":[["COLLECT_FERTILIZER"],["CARE"],["PLACE","FERTILIZER"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["PLACE","FERTILIZER"],["NORTH"],["WEST"]],"market":[["SELL","FERTILIZER",2],["BUY_PRODUCT","FERTILIZER",1]]},{"farmer":["EAST"],"hands":[["EAST"],["COLLECT_FERTILIZER"],["PICKUP","COW"],["EAST"],["SOUTH"],["EAST"],["NORTH"],["NORTH"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["PLACE","FERTILIZER"],"hands":[["PLACE","FERTILIZER"],["SOUTH"],["EAST"],["SOUTH"],["PLACE","FERTILIZER"],["EAST"],["NORTH"],["NORTH"]],"market":[["SELL","FERTILIZER",3],["BUY_ANIMAL","COW",1]]},{"farmer":["WEST"],"hands":[["NORTH"],["SOUTH"],["PLACE","COW"],["PLACE","FERTILIZER"],["WEST"],["EAST"],["WATER"],["WATER"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[["NORTH"],["PLACE","FERTILIZER"],["CARE"],["EAST"],["NORTH"],["EAST"],["WEST"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["NORTH"],["WEST"],["WEST"],["EAST"],["NORTH"],["EAST"],["PLANT","STRAWBERRY"],["WATER"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WATER"],"hands":[["WATER"],["WEST"],["PICKUP","COW"],["EAST"],["WATER"],["NORTH"],["WATER"],["NORTH"]],"market":[]},{"farmer":["WEST"],"hands":[["NORTH"],["WEST"],["NORTH"],["EAST"],["WEST"],["PLANT","STRAWBERRY"],["SOUTH"],["PLANT","WHEAT"]],"market":[]},{"farmer":["WATER"],"hands":[["WATER"],["NORTH"],["NORTH"],["NORTH"],["WATER"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["EAST"],["EAST"],["PLACE","COW"],["NORTH"],["WEST"],["NORTH"],["EAST"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["WATER"],["EAST"],["CARE"],["NORTH"],["WATER"],["PLANT","STRAWBERRY"],["WATER"],["PLANT","WHEAT"]],"market":[]},{"farmer":["WEST"],"hands":[["EAST"],["NORTH"],["EAST"],["PLANT","STRAWBERRY"],["EAST"],["WATER"],["EAST"],["WATER"]],"market":[]},{"farmer":["WATER"],"hands":[["WATER"],["NORTH"],["EAST"],["WATER"],["EAST"],["NORTH"],["WATER"],["EAST"]],"market":[]},{"farmer":["PASS"],"hands":[["EAST"],["NORTH"],["WEST"],["WEST"],["EAST"],["PLANT","WHEAT"],["NORTH"],["PASS"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["WATER"],["WEST"],["WATER"],["EAST"],["WATER"],["WATER"],["PASS"]],"market":[]},{"farmer":["PASS"],"hands":[["EAST"],["PASS"],["SOUTH"],["PASS"],["SOUTH"],["NORTH"],["PASS"],["PASS"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["PASS"],["CARE"],["PASS"],["COLLECT_FERTILIZER"],["PLANT","WHEAT"],["PASS"],["PASS"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"],["PASS"],["CARE"],["WATER"],["PASS"],["PASS"]],"market":[]},{"farmer":["WEST"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["NORTH"],["PASS"],["NORTH"],["NORTH"],["PICKUP","WHEAT",2],["PICKUP","WHEAT",2],["WEST"],["WEST"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["HARVEST"],["PICKUP","WHEAT"],["NORTH"],["PICKUP","WHEAT"],["FEED"],["FEED"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["EAST"],"hands":[["SOUTH"],["NORTH"],["NORTH"],["NORTH"],["HARVEST"],["CARE"],["NORTH"],["PICKUP","WHEAT",2]],"market":[]},{"farmer":["EAST"],"hands":[["DROP"],["NORTH"],["NORTH"],["FEED"],["PLACE","MILK",6],["COLLECT_FERTILIZER"],["NORTH"],["WEST"]],"market":[["SELL","MILK",27],["BUY_PRODUCT","WHEAT",6],[],["BUY_ANIMAL","SHEEP",2],["BUY_PRODUCT","FERTILIZER",1]]},{"farmer":["DROP"],"hands":[["CARE"],["FEED"],["WATER"],["CARE"],["PICKUP","SHEEP"],["PLACE","FERTILIZER"],["WATER"],["FEED"]],"market":[["SELL","FERTILIZER",3]]},{"farmer":["PICKUP","WHEAT",2],"hands":[["COLLECT_FERTILIZER"],["CARE"],["EAST"],["COLLECT_FERTILIZER"],["EAST"],["PICKUP","SHEEP"],["NORTH"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["NORTH"],"hands":[["PICKUP","WHEAT"],["COLLECT_FERTILIZER"],["WATER"],["SOUTH"],["EAST"],["EAST"],["WATER"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["NORTH"],"hands":[["NORTH"],["EAST"],["EAST"],["DROP"],["NORTH"],["EAST"],["WEST"],["EAST"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["FEED"],"hands":[["FEED"],["EAST"],["WATER"],["EAST"],["PASS"],["PLACE","SHEEP"],["WATER"],["EAST"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["CARE"],"hands":[["CARE"],["WATER"],["NORTH"],["EAST"],["PASS"],["CARE"],["SOUTH"],["EAST"]],"market":[["BUY_PRODUCT","WHEAT",3],["BUY_SEED","STRAWBERRY",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["EAST"],["PASS"],["FEED"],["WATER"],["FEED"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["WEST"],"hands":[["EAST"],["NORTH"],["HARVEST"],["NORTH"],["PASS"],["NORTH"],["WEST"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["SOUTH"],"hands":[["SOUTH"],["WATER"],["PLANT","STRAWBERRY"],["WATER"],["PASS"],["WATER"],["WATER"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["FEED"],"hands":[["SOUTH"],["HARVEST"],["WATER"],["NORTH"],["PASS"],["NORTH"],["SOUTH"],["WEST"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["CARE"],"hands":[["DROP"],["PLANT","STRAWBERRY"],["WEST"],["WATER"],["PASS"],["NORTH"],["WATER"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["NORTH"],["WATER"],["WATER"],["EAST"],["PASS"],["WATER"],["SOUTH"],["SOUTH"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["WEST"],"hands":[["PASS"],["EAST"],["HARVEST"],["WATER"],["PASS"],["HARVEST"],["WATER"],["DROP"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["PASS"],["PASS"],["PLANT","STRAWBERRY"],["SOUTH"],["PASS"],["PLANT","STRAWBERRY"],["WEST"],["NORTH"]],"market":[]},{"farmer":["SOUTH"],"hands":[["PASS"],["PASS"],["WATER"],["WATER"],["PASS"],["WATER"],["WATER"],["PASS"]],"market":[]},{"farmer":["CARE"],"hands":[["PASS"],["WATER"],["WEST"],["SOUTH"],["PASS"],["EAST"],["NORTH"],["PASS"]],"market":[["SELL","WHEAT",2]]},{"farmer":["EAST"],"hands":[["PASS"],["HARVEST"],["WATER"],["WATER"],["PLACE","SHEEP"],["WATER"],["NORTH"],["PASS"]],"market":[]},{"farmer":["EAST"],"hands":[["PASS"],["PLANT","WHEAT"],["WEST"],["WEST"],["CARE"],["EAST"],["NORTH"],["PASS"]],"market":[["SELL","FERTILIZER",3]]},{"farmer":["DROP"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"],["FEED"],["WATER"],["WATER"],["PASS"]],"market":[["SELL","FERTILIZER",4]]},{"farmer":["PICKUP","WHEAT",2],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["FEED"],"hands":[["PICKUP","WHEAT"],["WEST"],["NORTH"],["PICKUP","WHEAT",2],["PASS"],["EAST"],["NORTH"],["PICKUP","WHEAT",3]],"market":[["SELL","FERTILIZER",1],[],["BUY_ANIMAL","SHEEP",1],["BUY_PRODUCT","FERTILIZER",1]]},{"farmer":["CARE"],"hands":[["EAST"],["NORTH"],["PICKUP","WHEAT",2],["WEST"],["PICKUP","WHEAT",2],["NORTH"],["PICKUP","WHEAT"],["NORTH"]],"market":[["SELL","FERTILIZER",1],["BUY_PRODUCT","WHEAT",3]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["FEED"],["NORTH"],["EAST"],["FEED"],["FEED"],["PASS"],["EAST"],["FEED"]],"market":[]},{"farmer":["PLACE","FERTILIZER"],"hands":[["CARE"],["HARVEST"],["COLLECT_FERTILIZER"],["CARE"],["CARE"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["WEST"],"hands":[["EAST"],["EAST"],["WEST"],["COLLECT_FERTILIZER"],["EAST"],["WEST"],["FEED"],["COLLECT_FERTILIZER"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[["EAST"],["EAST"],["PICKUP","SHEEP"],["HARVEST"],["NORTH"],["DROP"],["CARE"],["EAST"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["FEED"],"hands":[["WATER"],["SOUTH"],["PLACE","FERTILIZER"],["EAST"],["COLLECT_FERTILIZER"],["NORTH"],["EAST"],["FEED"]],"market":[]},{"farmer":["CARE"],"hands":[["EAST"],["DROP"],["NORTH"],["PLACE","WOOL",4],["WEST"],["NORTH"],["WATER"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["WEST"],"hands":[["WATER"],["WEST"],["NORTH"],["SOUTH"],["NORTH"],["NORTH"],["NORTH"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["WATER"],"hands":[["NORTH"],["WEST"],["FEED"],["PLACE","FERTILIZER"],["PASS"],["WATER"],["WATER"],["WEST"]],"market":[["SELL","WOOL",24],["BUY_ANIMAL","SHEEP",1]]},{"farmer":["WEST"],"hands":[["WATER"],["WEST"],["CARE"],["WEST"],["COLLECT_FERTILIZER"],["NORTH"],["NORTH"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["NORTH"],["NORTH"],["EAST"],["WEST"],["EAST"],["WATER"],["WATER"],["NORTH"]],"market":[["BUY_PRODUCT","WHEAT",3],["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["WATER"],["WATER"],["PLACE","SHEEP"],["NORTH"],["EAST"],["WEST"],["EAST"],["WATER"]],"market":[]},{"farmer":["WATER"],"hands":[["NORTH"],["WEST"],["CARE"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["WATER"],["SOUTH"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["NORTH"],"hands":[["WATER"],["WATER"],["FEED"],["EAST"],["SOUTH"],["WEST"],["SOUTH"],["FEED"]],"market":[]},{"farmer":["WATER"],"hands":[["HARVEST"],["NORTH"],["NORTH"],["EAST"],["FEED"],["WATER"],["WATER"],["CARE"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["PLANT","WHEAT"],["WATER"],["WATER"],["PLACE","FERTILIZER"],["CARE"],["WEST"],["SOUTH"],["COLLECT_FERTILIZER"]],"market":[["SELL","FERTILIZER",3],["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WATER"],["EAST"],["NORTH"],["NORTH"],["COLLECT_FERTILIZER"],["WATER"],["WATER"],["EAST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["NORTH"],["WATER"],["WATER"],["NORTH"],["WEST"],["WEST"],["WEST"],["EAST"]],"market":[]},{"farmer":["PLANT","WHEAT"],"hands":[["WATER"],["NORTH"],["WEST"],["FEED"],["WEST"],["WATER"],["NORTH"],["SOUTH"]],"market":[]},{"farmer":["WATER"],"hands":[["HARVEST"],["WATER"],["WATER"],["CARE"],["DROP"],["HARVEST"],["NORTH"],["DROP"]],"market":[["SELL","FERTILIZER",3]]},{"farmer":["EAST"],"hands":[["PLANT","WHEAT"],["EAST"],["SOUTH"],["COLLECT_FERTILIZER"],["PASS"],["PLANT","WHEAT"],["NORTH"],["PASS"]],"market":[["SELL","FERTILIZER",5]]},{"farmer":["WATER"],"hands":[["WATER"],["WATER"],["WATER"],["PASS"],["PASS"],["WATER"],["WATER"],["PASS"]],"market":[]},{"farmer":["NORTH"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["WEST"],["WEST"],["WEST"],["WEST"],["NORTH"],["WEST"],["WEST"],["NORTH"],["WEST"]],"market":[["HIRE"],["HIRE"],["BUY_ANIMAL","GOOSE",2],["BUY_PRODUCT","FERTILIZER",3]]},{"farmer":["WEST"],"hands":[["WEST"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["PICKUP","WHEAT",5],["WEST"],["NORTH"],["NORTH"]],"market":[["SELL","FERTILIZER",3],["BUY_PRODUCT","WHEAT",5]]},{"farmer":["WEST"],"hands":[["WEST"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["FEED"],["WEST"],["WEST"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["WEST"],["NORTH"],["NORTH"],["NORTH"],["NORTH"],["WEST"],["WEST"],["CARE"],["WATER"],["PICKUP","WHEAT",5],["NORTH"]],"market":[["BUY_PRODUCT","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["WATER"],["WATER"],["WEST"],["NORTH"],["WATER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["FEED"],["WEST"]],"market":[]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["HARVEST"],["WATER"],["NORTH"],["HARVEST"],["HARVEST"],["HARVEST"],["NORTH"],["EAST"],["CARE"],["WEST"]],"market":[]},{"farmer":["EAST"],"hands":[["EAST"],["SOUTH"],["HARVEST"],["WEST"],["SOUTH"],["SOUTH"],["SOUTH"],["FEED"],["EAST"],["COLLECT_FERTILIZER"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["EAST"],["SOUTH"],["SOUTH"],["WATER"],["SOUTH"],["SOUTH"],["EAST"],["CARE"],["EAST"],["NORTH"],["HARVEST"]],"market":[]},{"farmer":["EAST"],"hands":[["EAST"],["SOUTH"],["SOUTH"],["HARVEST"],["SOUTH"],["EAST"],["EAST"],["COLLECT_FERTILIZER"],["PLACE","MELON",6],["FEED"],["SOUTH"]],"market":[["SELL","MELON",6],["BUY_PRODUCT","WHEAT",14],["BUY_PRODUCT","FERTILIZER",5]]},{"farmer":["PLACE","MELON",6],"hands":[["EAST"],["PLACE","MELON",6],["EAST"],["SOUTH"],["SOUTH"],["PLACE","MELON",6],["PLACE","MELON",6],["EAST"],["PICKUP","WHEAT",3],["CARE"],["SOUTH"]],"market":[["SELL","MELON",24],["BUY_PRODUCT","WHEAT",5]]},{"farmer":["NORTH"],"hands":[["PLACE","MELON",6],["NORTH"],["EAST"],["SOUTH"],["PLACE","MELON",6],["NORTH"],["NORTH"],["FEED"],["WEST"],["COLLECT_FERTILIZER"],["SOUTH"]],"market":[["SELL","MELON",12]]},{"farmer":["NORTH"],"hands":[["PICKUP","SHEEP"],["NORTH"],["PLACE","MELON",6],["SOUTH"],["NORTH"],["WEST"],["NORTH"],["CARE"],["FEED"],["NORTH"],["EAST"]],"market":[["SELL","MELON",6],["BUY_PRODUCT","WHEAT",5]]},{"farmer":["HARVEST"],"hands":[["PICKUP","WHEAT"],["NORTH"],["PICKUP","GOOSE"],["SOUTH"],["NORTH"],["WEST"],["WEST"],["COLLECT_FERTILIZER"],["CARE"],["FEED"],["PLACE","MELON",6]],"market":[["SELL","MELON",6],["SELL","MILK",6]]},{"farmer":["SOUTH"],"hands":[["WEST"],["NORTH"],["PICKUP","WHEAT"],["EAST"],["NORTH"],["BUILD_COOP"],["WEST"],["SOUTH"],["COLLECT_FERTILIZER"],["CARE"],["PICKUP","GOOSE"]],"market":[["BUY_PRODUCT","WHEAT",5],["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["WEST"],["BUILD_COOP"],["NORTH"],["PLACE","MELON",6],["WEST"],["WEST"],["PLANT","WHEAT"],["FEED"],["NORTH"],["COLLECT_FERTILIZER"],["PICKUP","WHEAT"]],"market":[["SELL","MELON",6],["BUY_SEED","WHEAT",2]]},{"farmer":["SOUTH"],"hands":[["WEST"],["EAST"],["NORTH"],["HARVEST"],["PLANT","WHEAT"],["PLANT","WHEAT"],["WATER"],["CARE"],["FEED"],["EAST"],["NORTH"]],"market":[["BUY_PRODUCT","WHEAT",5]]},{"farmer":["HARVEST"],"hands":[["BUILD_PASTURE"],["EAST"],["WEST"],["PASS"],["WATER"],["WATER"],["NORTH"],["COLLECT_FERTILIZER"],["CARE"],["FEED"],["NORTH"]],"market":[["SELL","FERTILIZER",5]]},{"farmer":["PLACE","MILK",12],"hands":[["PLACE","SHEEP"],["EAST"],["BUILD_COOP"],["PASS"],["NORTH"],["NORTH"],["WATER"],["EAST"],["COLLECT_FERTILIZER"],["CARE"],["NORTH"]],"market":[["SELL","MILK",21],["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["FEED"],["EAST"],["PLACE","GOOSE"],["PASS"],["PLANT","WHEAT"],["WATER"],["HARVEST"],["FEED"],["SOUTH"],["COLLECT_FERTILIZER"],["BUILD_COOP"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["CARE"],["WATER"],["FEED"],["PASS"],["WATER"],["HARVEST"],["PLANT","WHEAT"],["CARE"],["WEST"],["EAST"],["PLACE","GOOSE"]],"market":[["SELL","WHEAT",2],["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["WEST"],["PASS"],["CARE"],["PASS"],["PASS"],["PLANT","WHEAT"],["WATER"],["COLLECT_FERTILIZER"],["FEED"],["FEED"],["FEED"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["PASS"],"hands":[["PLANT","WHEAT"],["PASS"],["PASS"],["PASS"],["PASS"],["WATER"],["PASS"],["PASS"],["CARE"],["CARE"],["CARE"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["PASS"]],"market":[]},{"farmer":["SOUTH"],"hands":[],"market":[["SELL","MELON",12],["SELL","FERTILIZER",13],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["NORTH"],"hands":[["WEST"],["EAST"],["SOUTH"],["PASS"],["COLLECT_FERTILIZER"]],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],[],["BUY_LAND"],["BUY_ANIMAL","GOOSE",1],["BUY_PRODUCT","FERTILIZER",1],["BUY_SEED","STRAWBERRY",2]]},{"farmer":["PICKUP","WHEAT",2],"hands":[["WEST"],["EAST"],["WEST"],["EAST"],["CARE"],["NORTH"],["WEST"],["PICKUP","WHEAT"],["WEST"],["NORTH"]],"market":[["SELL","FERTILIZER",1],["BUY_PRODUCT","WHEAT",3],["BUY_SEED","STRAWBERRY",1]]},{"farmer":["FEED"],"hands":[["HARVEST"],["EAST"],["WEST"],["PICKUP","WHEAT"],["PICKUP","WHEAT",3],["PICKUP","WHEAT",2],["PLANT","STRAWBERRY"],["NORTH"],["WEST"],["PICKUP","WHEAT",2]],"market":[["SELL","MILK",6]]},{"farmer":["CARE"],"hands":[["EAST"],["WATER"],["NORTH"],["EAST"],["EAST"],["PICKUP","GOOSE"],["WATER"],["COLLECT_FERTILIZER"],["PASS"],["WEST"]],"market":[["BUY_PRODUCT","WHEAT",3],["BUY_SEED","STRAWBERRY",2]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["EAST"],["EAST"],["PASS"],["EAST"],["COLLECT_FERTILIZER"],["NORTH"],["SOUTH"],["CARE"],["PASS"],["NORTH"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["WEST"],"hands":[["DROP"],["WATER"],["PASS"],["FEED"],["CARE"],["FEED"],["PLANT","STRAWBERRY"],["NORTH"],["PASS"],["NORTH"]],"market":[["SELL","MILK",14],["BUY_PRODUCT","WHEAT",3],["BUY_SEED","STRAWBERRY",1]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",2],["NORTH"],["PLANT","STRAWBERRY"],["CARE"],["NORTH"],["CARE"],["WATER"],["NORTH"],["PLANT","STRAWBERRY"],["FEED"]],"market":[["BUY_SEED","STRAWBERRY",2]]},{"farmer":["FEED"],"hands":[["WEST"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["FEED"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["WATER"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3],["BUY_SEED","STRAWBERRY",1]]},{"farmer":["CARE"],"hands":[["FEED"],["NORTH"],["SOUTH"],["NORTH"],["CARE"],["WEST"],["PLANT","STRAWBERRY"],["NORTH"],["WEST"],["COLLECT_FERTILIZER"]],"market":[["BUY_SEED","STRAWBERRY",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["WATER"],["PLANT","STRAWBERRY"],["WATER"],["COLLECT_FERTILIZER"],["WEST"],["WATER"],["WATER"],["PLANT","STRAWBERRY"],["WEST"]],"market":[["BUY_PRODUCT","WHEAT",3],["BUY_SEED","STRAWBERRY",1],["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["NORTH"],["NORTH"],["PLACE","GOOSE"],["SOUTH"],["WEST"],["WATER"],["FEED"]],"market":[]},{"farmer":["SOUTH"],"hands":[["WEST"],["WATER"],["SOUTH"],["WATER"],["FEED"],["CARE"],["PLANT","STRAWBERRY"],["SOUTH"],["SOUTH"],["CARE"]],"market":[["BUY_PRODUCT","WHEAT",3],["BUY_SEED","WHEAT",2]]},{"farmer":["PLANT","STRAWBERRY"],"hands":[["WEST"],["NORTH"],["PLANT","STRAWBERRY"],["NORTH"],["CARE"],["FEED"],["WATER"],["FEED"],["PLANT","STRAWBERRY"],["COLLECT_FERTILIZER"]],"market":[[],["BUY_SEED","WHEAT",2]]},{"farmer":["WATER"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["WEST"],["WEST"],["CARE"],["WATER"],["WEST"]],"market":[["BUY_PRODUCT","WHEAT",3]]},{"farmer":["SOUTH"],"hands":[["PASS"],["HARVEST"],["SOUTH"],["NORTH"],["WEST"],["WEST"],["WEST"],["COLLECT_FERTILIZER"],["WEST"],["WATER"]],"market":[["BUY_SEED","STRAWBERRY",1],["BUY_SEED","WHEAT",1]]},{"farmer":["PLANT","STRAWBERRY"],"hands":[["FEED"],["WEST"],["PLANT","STRAWBERRY"],["WATER"],["COLLECT_FERTILIZER"],["WATER"],["PLANT","WHEAT"],["WEST"],["PLANT","WHEAT"],["NORTH"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["WATER"],"hands":[["CARE"],["WATER"],["WATER"],["WEST"],["CARE"],["NORTH"],["WATER"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["WEST"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["SOUTH"],["WATER"],["WEST"],["WATER"],["SOUTH"],["NORTH"],["SOUTH"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["PLANT","WHEAT"],"hands":[["WEST"],["SOUTH"],["PLANT","WHEAT"],["SOUTH"],["WEST"],["NORTH"],["PLANT","WHEAT"],["WATER"],["PLANT","WHEAT"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WATER"],["SOUTH"],["WATER"],["WATER"],["SOUTH"],["WATER"],["WATER"],["WEST"],["WATER"],["SOUTH"]],"market":[]},{"farmer":["SOUTH"],"hands":[["SOUTH"],["WATER"],["EAST"],["EAST"],["FEED"],["NORTH"],["WEST"],["WATER"],["SOUTH"],["WATER"]],"market":[[],["BUY_SEED","WHEAT",1]]},{"farmer":["PLANT","WHEAT"],"hands":[["PLANT","WHEAT"],["SOUTH"],["PLANT","WHEAT"],["EAST"],["CARE"],["WATER"],["PLANT","WHEAT"],["WEST"],["PLANT","WHEAT"],["SOUTH"]],"market":[]},{"farmer":["WATER"],"hands":[["WATER"],["WATER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["WATER"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["NORTH"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",5],["PICKUP","WHEAT",5],["NORTH"],["NORTH"],["PICKUP","WHEAT",5],["NORTH"],["NORTH"],["SOUTH"],["SOUTH"]],"market":[["SELL","FERTILIZER",14]]},{"farmer":["HARVEST"],"hands":[["WEST"],["FEED"],["WEST"],["NORTH"],["FEED"],["NORTH"],["NORTH"],["SOUTH"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["FEED"],["CARE"],["HARVEST"],["NORTH"],["CARE"],["NORTH"],["NORTH"],["SOUTH"],["SOUTH"]],"market":[]},{"farmer":["EAST"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["EAST"],["NORTH"],["COLLECT_FERTILIZER"],["EAST"],["NORTH"],["SOUTH"],["WEST"]],"market":[]},{"farmer":["PLACE","WOOL",4],"hands":[["COLLECT_FERTILIZER"],["NORTH"],["PLACE","WOOL",4],["WEST"],["NORTH"],["EAST"],["NORTH"],["WEST"],["WATER"]],"market":[["SELL","WOOL",24]]},{"farmer":["NORTH"],"hands":[["NORTH"],["FEED"],["EAST"],["WEST"],["FEED"],["EAST"],["WEST"],["WEST"],["EAST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["FEED"],["CARE"],["PICKUP","WHEAT",2],["WEST"],["CARE"],["EAST"],["WEST"],["WEST"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["EAST"],["WEST"],["COLLECT_FERTILIZER"],["WATER"],["WEST"],["WEST"],["SOUTH"]],"market":[]},{"farmer":["HARVEST"],"hands":[["COLLECT_FERTILIZER"],["NORTH"],["FEED"],["WEST"],["NORTH"],["HARVEST"],["WEST"],["WEST"],["WATER"]],"market":[["BUY_SEED","WHEAT",3]]},{"farmer":["PLACE","MILK",6],"hands":[["WEST"],["FEED"],["CARE"],["WATER"],["FEED"],["PLANT","WHEAT"],["PLANT","WHEAT"],["PLANT","WHEAT"],["SOUTH"]],"market":[]},{"farmer":["SOUTH"],"hands":[["FEED"],["CARE"],["COLLECT_FERTILIZER"],["HARVEST"],["CARE"],["WATER"],["WATER"],["WATER"],["SOUTH"]],"market":[["SELL","FERTILIZER",2],["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["EAST"],["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["NORTH"],["EAST"],["NORTH"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["COLLECT_FERTILIZER"],["EAST"],["FEED"],["WATER"],["NORTH"],["PLANT","WHEAT"],["EAST"],["WATER"],["WEST"]],"market":[]},{"farmer":["WEST"],"hands":[["SOUTH"],["FEED"],["CARE"],["EAST"],["FEED"],["WATER"],["EAST"],["NORTH"],["WATER"]],"market":[]},{"farmer":["WATER"],"hands":[["FEED"],["CARE"],["COLLECT_FERTILIZER"],["EAST"],["CARE"],["WEST"],["WATER"],["WATER"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["PASS"],["WATER"],["COLLECT_FERTILIZER"],["PLANT","WHEAT"],["HARVEST"],["NORTH"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["COLLECT_FERTILIZER"],["SOUTH"],["PASS"],["HARVEST"],["SOUTH"],["WATER"],["PLANT","WHEAT"],["WATER"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WEST"],["FEED"],["PASS"],["PLANT","WHEAT"],["WEST"],["PASS"],["WATER"],["SOUTH"],["WATER"]],"market":[["SELL","EGG",6]]},{"farmer":["NORTH"],"hands":[["FEED"],["CARE"],["PASS"],["WATER"],["FEED"],["PASS"],["SOUTH"],["EAST"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["PASS"],["SOUTH"],["CARE"],["PASS"],["WATER"],["WATER"],["WATER"]],"market":[["SELL","WHEAT",7]]},{"farmer":["PASS"],"hands":[["COLLECT_FERTILIZER"],["PASS"],["PASS"],["WATER"],["COLLECT_FERTILIZER"],["PASS"],["PASS"],["PASS"],["EAST"]],"market":[]},{"farmer":["PASS"],"hands":[["NORTH"],["PASS"],["PASS"],["WEST"],["PASS"],["PASS"],["PASS"],["PASS"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["WATER"],["PASS"],["PASS"],["WATER"],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"]],"market":[]},{"farmer":["NORTH"],"hands":[],"market":[["SELL","FERTILIZER",17],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["PASS"],"hands":[["PICKUP","WHEAT"],["SOUTH"],["EAST"],["NORTH"],["SOUTH"],["WEST"],["PICKUP","WHEAT",3],["NORTH"]],"market":[["HIRE"],[]]},{"farmer":["NORTH"],"hands":[["WEST"],["WEST"],["NORTH"],["WEST"],["SOUTH"],["WEST"],["WEST"],["PICKUP","WHEAT",4],["PICKUP","WHEAT",2]],"market":[]},{"farmer":["HARVEST"],"hands":[["NORTH"],["WEST"],["PICKUP","WHEAT",2],["PICKUP","WHEAT"],["SOUTH"],["WEST"],["NORTH"],["FEED"],["NORTH"]],"market":[]},{"farmer":["SOUTH"],"hands":[["NORTH"],["WEST"],["NORTH"],["EAST"],["SOUTH"],["HARVEST"],["FEED"],["CARE"],["FEED"]],"market":[]},{"farmer":["SOUTH"],"hands":[["FEED"],["SOUTH"],["FEED"],["EAST"],["WATER"],["EAST"],["CARE"],["COLLECT_FERTILIZER"],["CARE"]],"market":[]},{"farmer":["DROP"],"hands":[["CARE"],["WATER"],["CARE"],["EAST"],["SOUTH"],["EAST"],["COLLECT_FERTILIZER"],["EAST"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["PICKUP","WHEAT",3],"hands":[["COLLECT_FERTILIZER"],["WEST"],["COLLECT_FERTILIZER"],["PASS"],["WATER"],["SOUTH"],["NORTH"],["FEED"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["NORTH"],["PASS"],["NORTH"],["FEED"],["WEST"],["DROP"],["FEED"],["CARE"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["WATER"],["PASS"],["FEED"],["CARE"],["WATER"],["WEST"],["CARE"],["COLLECT_FERTILIZER"],["FEED"]],"market":[["SELL","MILK",14]]},{"farmer":["FEED"],"hands":[["HARVEST"],["PASS"],["CARE"],["COLLECT_FERTILIZER"],["WEST"],["WEST"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"]],"market":[]},{"farmer":["CARE"],"hands":[["PLANT","WHEAT"],["WATER"],["COLLECT_FERTILIZER"],["EAST"],["WATER"],["WEST"],["WEST"],["FEED"],["COLLECT_FERTILIZER"]],"market":[["SELL","MILK",6],["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WATER"],["SOUTH"],["NORTH"],["WATER"],["WEST"],["WATER"],["FEED"],["CARE"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["WEST"],["WATER"],["WATER"],["EAST"],["WATER"],["WEST"],["CARE"],["COLLECT_FERTILIZER"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["WATER"],["EAST"],["NORTH"],["WATER"],["WEST"],["WATER"],["COLLECT_FERTILIZER"],["NORTH"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["SOUTH"],["WATER"],["WATER"],["NORTH"],["NORTH"],["NORTH"],["WEST"],["FEED"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["WATER"],["EAST"],["EAST"],["WATER"],["WATER"],["WATER"],["WATER"],["CARE"],["WATER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["HARVEST"],["WATER"],["WATER"],["NORTH"],["HARVEST"],["HARVEST"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"]],"market":[]},{"farmer":["WEST"],"hands":[["PLANT","WHEAT"],["SOUTH"],["EAST"],["WATER"],["PLANT","WHEAT"],["PLANT","WHEAT"],["PLANT","WHEAT"],["EAST"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["WATER"],["WATER"],["WATER"],["WEST"],["WATER"],["WATER"],["WATER"],["WATER"],["SOUTH"]],"market":[]},{"farmer":["FEED"],"hands":[["WEST"],["WEST"],["SOUTH"],["SOUTH"],["NORTH"],["NORTH"],["SOUTH"],["EAST"],["WATER"]],"market":[["SELL","WHEAT",7]]},{"farmer":["CARE"],"hands":[["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["FEED"],["WATER"],["SOUTH"]],"market":[["SELL","MILK",6]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["HARVEST"],["WEST"],["WEST"],["WEST"],["NORTH"],["NORTH"],["CARE"],["NORTH"],["PLANT","WHEAT"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["WATER"],["WATER"]],"market":[]},{"farmer":["PICKUP","FERTILIZER",4],"hands":[],"market":[["BUY_SEED","WHEAT",9],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["NORTH"],"hands":[["HARVEST"],["WEST"],["EAST"],["PICKUP","WHEAT",5],["EAST"],["NORTH"],["WEST"],["PICKUP","WHEAT",5],["NORTH"]],"market":[["SELL","FERTILIZER",10],["HIRE"],["BUY_PRODUCT","FERTILIZER",2],["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[["PLACE","MILK",6],["WATER"],["NORTH"],["FEED"],["EAST"],["NORTH"],["WATER"],["HARVEST"],["HARVEST"],["NORTH"]],"market":[["SELL","MILK",14],["SELL","FERTILIZER",2]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",5],["SOUTH"],["NORTH"],["CARE"],["HARVEST"],["HARVEST"],["SOUTH"],["PLACE","MILK",3],["SOUTH"],["PICKUP","WHEAT",2]],"market":[["SELL","MILK",6]]},{"farmer":["WEST"],"hands":[["FEED"],["WATER"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"],["SOUTH"],["WATER"],["WEST"],["PLACE","MILK",6],["EAST"]],"market":[["SELL","MILK",14]]},{"farmer":["WEST"],"hands":[["CARE"],["WEST"],["WEST"],["NORTH"],["WEST"],["PLACE","MILK",3],["SOUTH"],["FEED"],["WEST"],["FEED"]],"market":[["SELL","MILK",6]]},{"farmer":["FERTILIZE"],"hands":[["COLLECT_FERTILIZER"],["WEST"],["SOUTH"],["FEED"],["PLACE","WOOL",6],["WEST"],["WATER"],["CARE"],["WEST"],["CARE"]],"market":[["SELL","WOOL",18]]},{"farmer":["WATER"],"hands":[["NORTH"],["SOUTH"],["PLACE","WOOL",6],["CARE"],["WEST"],["WEST"],["WEST"],["COLLECT_FERTILIZER"],["WEST"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["NORTH"],"hands":[["FEED"],["WATER"],["WEST"],["COLLECT_FERTILIZER"],["WEST"],["SOUTH"],["WEST"],["NORTH"],["SOUTH"],["EAST"]],"market":[]},{"farmer":["FERTILIZE"],"hands":[["CARE"],["HARVEST"],["SOUTH"],["NORTH"],["WEST"],["SOUTH"],["SOUTH"],["FEED"],["WATER"],["FEED"]],"market":[]},{"farmer":["WATER"],"hands":[["COLLECT_FERTILIZER"],["PLANT","WHEAT"],["SOUTH"],["FEED"],["WEST"],["WATER"],["WATER"],["CARE"],["WEST"],["CARE"]],"market":[]},{"farmer":["EAST"],"hands":[["NORTH"],["WATER"],["SOUTH"],["CARE"],["NORTH"],["SOUTH"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["NORTH"],"hands":[["FEED"],["SOUTH"],["SOUTH"],["COLLECT_FERTILIZER"],["WATER"],["WATER"],["PLANT","WHEAT"],["WEST"],["SOUTH"],["EAST"]],"market":[]},{"farmer":["FERTILIZE"],"hands":[["CARE"],["WATER"],["WATER"],["HARVEST"],["NORTH"],["WEST"],["WATER"],["FEED"],["WATER"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["WEST"],["NORTH"],["WATER"],["NORTH"],["SOUTH"],["CARE"],["HARVEST"],["NORTH"]],"market":[]},{"farmer":["EAST"],"hands":[["EAST"],["PLANT","WHEAT"],["WATER"],["FEED"],["EAST"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["PLANT","WHEAT"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["FEED"],["WATER"],["NORTH"],["HARVEST"],["WATER"],["NORTH"],["HARVEST"],["SOUTH"],["WATER"],["NORTH"]],"market":[]},{"farmer":["FERTILIZE"],"hands":[["CARE"],["SOUTH"],["WATER"],["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["PLANT","WHEAT"],["FEED"],["SOUTH"],["WATER"]],"market":[]},{"farmer":["WATER"],"hands":[["COLLECT_FERTILIZER"],["WATER"],["PASS"],["WEST"],["WATER"],["WEST"],["WATER"],["CARE"],["WATER"],["EAST"]],"market":[]},{"farmer":["WEST"],"hands":[["SOUTH"],["HARVEST"],["PASS"],["SOUTH"],["EAST"],["WATER"],["EAST"],["COLLECT_FERTILIZER"],["HARVEST"],["WATER"]],"market":[["SELL","EGG",9]]},{"farmer":["WEST"],"hands":[["FEED"],["PLANT","WHEAT"],["PASS"],["FEED"],["WATER"],["NORTH"],["WATER"],["WEST"],["PLANT","WHEAT"],["SOUTH"]],"market":[["SELL","WHEAT",9]]},{"farmer":["WATER"],"hands":[["CARE"],["WATER"],["PASS"],["HARVEST"],["NORTH"],["WATER"],["EAST"],["FEED"],["WATER"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["WEST"],["PASS"],["COLLECT_FERTILIZER"],["WATER"],["EAST"],["WATER"],["HARVEST"],["SOUTH"],["PASS"]],"market":[]},{"farmer":["WATER"],"hands":[["PASS"],["WATER"],["PASS"],["CARE"],["PASS"],["CARE"],["PASS"],["COLLECT_FERTILIZER"],["WATER"],["PASS"]],"market":[]},{"farmer":["PICKUP","WHEAT",4],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["PASS"],["EAST"],["WEST"],["PASS"],["PICKUP","WHEAT"],["NORTH"],["PASS"],["WEST"]],"market":[["SELL","FERTILIZER",7],["HIRE"],["HIRE"],[],["BUY_PRODUCT","FERTILIZER",4]]},{"farmer":["WEST"],"hands":[["WEST"],["NORTH"],["WATER"],["PICKUP","WHEAT",3],["PICKUP","FERTILIZER",3],["WEST"],["NORTH"],["EAST"],["SOUTH"],["PICKUP","WHEAT"]],"market":[["SELL","WOOL",9]]},{"farmer":["HARVEST"],"hands":[["PICKUP","WHEAT"],["PICKUP","WHEAT",2],["WEST"],["NORTH"],["EAST"],["WEST"],["PICKUP","WHEAT",3],["PICKUP","WHEAT"],["SOUTH"],["EAST"]],"market":[]},{"farmer":["FEED"],"hands":[["FEED"],["EAST"],["WEST"],["PASS"],["EAST"],["WEST"],["NORTH"],["WEST"],["SOUTH"],["EAST"]],"market":[]},{"farmer":["CARE"],"hands":[["CARE"],["FEED"],["WEST"],["FEED"],["FEED"],["WEST"],["NORTH"],["NORTH"],["SOUTH"],["NORTH"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["COLLECT_FERTILIZER"],["CARE"],["WEST"],["CARE"],["CARE"],["NORTH"],["HARVEST"],["NORTH"],["WATER"],["NORTH"]],"market":[]},{"farmer":["EAST"],"hands":[["EAST"],["COLLECT_FERTILIZER"],["WATER"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["HARVEST"],["FEED"],["PASS"],["HARVEST"],["HARVEST"]],"market":[["SELL","WHEAT",9],["BUY_SEED","WHEAT",1]]},{"farmer":["FEED"],"hands":[["NORTH"],["HARVEST"],["HARVEST"],["NORTH"],["EAST"],["NORTH"],["CARE"],["FEED"],["PLANT","WHEAT"],["PASS"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["WEST"],["PLANT","WHEAT"],["FEED"],["FERTILIZE"],["HARVEST"],["COLLECT_FERTILIZER"],["CARE"],["WATER"],["FEED"]],"market":[]},{"farmer":["CARE"],"hands":[["NORTH"],["WEST"],["WATER"],["CARE"],["WATER"],["EAST"],["WEST"],["COLLECT_FERTILIZER"],["WEST"],["CARE"]],"market":[]},{"farmer":["EAST"],"hands":[["FERTILIZE"],["PLACE","MILK",6],["NORTH"],["COLLECT_FERTILIZER"],["EAST"],["NORTH"],["SOUTH"],["HARVEST"],["WATER"],["COLLECT_FERTILIZER"]],"market":[["SELL","FERTILIZER",11],["SELL","FERTILIZER",2]]},{"farmer":["PLACE","MILK",3],"hands":[["WATER"],["EAST"],["WATER"],["NORTH"],["FERTILIZE"],["HARVEST"],["SOUTH"],["WEST"],["HARVEST"],["NORTH"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["WEST"],"hands":[["NORTH"],["EAST"],["EAST"],["FEED"],["WATER"],["EAST"],["PLACE","MILK",6],["WATER"],["PLANT","WHEAT"],["FERTILIZE"]],"market":[["SELL","MILK",21]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WATER"],["NORTH"],["FEED"],["CARE"],["NORTH"],["NORTH"],["EAST"],["NORTH"],["WATER"],["WATER"]],"market":[]},{"farmer":["NORTH"],"hands":[["EAST"],["FEED"],["CARE"],["COLLECT_FERTILIZER"],["WATER"],["HARVEST"],["NORTH"],["WATER"],["WEST"],["EAST"]],"market":[["SELL","MILK",33]]},{"farmer":["HARVEST"],"hands":[["WATER"],["CARE"],["COLLECT_FERTILIZER"],["NORTH"],["WEST"],["EAST"],["FEED"],["WEST"],["WEST"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["EAST"],["COLLECT_FERTILIZER"],["NORTH"],["DIG"],["FERTILIZE"],["EAST"],["CARE"],["NORTH"],["WEST"],["EAST"]],"market":[["SELL","EGG",6]]},{"farmer":["CARE"],"hands":[["WATER"],["EAST"],["WATER"],["PLANT","WHEAT"],["WATER"],["SOUTH"],["COLLECT_FERTILIZER"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["EAST"],["FERTILIZE"],["NORTH"],["WATER"],["NORTH"],["SOUTH"],["SOUTH"],["WEST"],["HARVEST"],["EAST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["WATER"],["WATER"],["WATER"],["WEST"],["FERTILIZE"],["SOUTH"],["FEED"],["WATER"],["PLANT","WHEAT"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["FEED"],"hands":[["HARVEST"],["NORTH"],["WEST"],["WATER"],["WATER"],["SOUTH"],["CARE"],["HARVEST"],["WATER"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["CARE"],"hands":[["PLANT","WHEAT"],["FERTILIZE"],["NORTH"],["SOUTH"],["EAST"],["DROP"],["COLLECT_FERTILIZER"],["PLANT","WHEAT"],["NORTH"],["HARVEST"]],"market":[["SELL","STRAWBERRY",8],["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["PASS"],["PASS"],["WATER"],["WATER"],["PLANT","WHEAT"]],"market":[]},{"farmer":["EAST"],"hands":[],"market":[["HIRE"],["BUY_SEED","WHEAT",8],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["EAST"],"hands":[["WEST"],["NORTH"],["NORTH"],["PICKUP","WHEAT",5],["NORTH"],["WEST"],["NORTH"],["PICKUP","WHEAT",5],["PICKUP","FERTILIZER",3]],"market":[["SELL","MILK",3],["HIRE"],["HIRE"],["BUY_PRODUCT","FERTILIZER",2]]},{"farmer":["EAST"],"hands":[["WEST"],["PICKUP","FERTILIZER"],["PICKUP","WHEAT",5],["FEED"],["NORTH"],["SOUTH"],["PICKUP","WHEAT",2],["WEST"],["EAST"],["WATER"],["WEST"]],"market":[["SELL","FERTILIZER",5]]},{"farmer":["NORTH"],"hands":[["NORTH"],["WEST"],["FEED"],["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["EAST"],["FEED"],["EAST"],["SOUTH"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["NORTH"],["WEST"],["CARE"],["HARVEST"],["HARVEST"],["WEST"],["FEED"],["CARE"],["EAST"],["WATER"],["WATER"]],"market":[["SELL","STRAWBERRY",8]]},{"farmer":["NORTH"],"hands":[["NORTH"],["WEST"],["COLLECT_FERTILIZER"],["NORTH"],["EAST"],["WATER"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["EAST"],["SOUTH"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["WATER"],["NORTH"],["NORTH"],["FEED"],["HARVEST"],["WEST"],["EAST"],["NORTH"],["NORTH"],["WATER"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["HARVEST"],["NORTH"],["FEED"],["CARE"],["WEST"],["WATER"],["FEED"],["FEED"],["FERTILIZE"],["WEST"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["PLANT","WHEAT"],["NORTH"],["CARE"],["COLLECT_FERTILIZER"],["SOUTH"],["WEST"],["CARE"],["CARE"],["WATER"],["WATER"],["WATER"]],"market":[["SELL","WHEAT",8]]},{"farmer":["SOUTH"],"hands":[["WATER"],["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["SOUTH"],["WATER"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["NORTH"],["WEST"],["NORTH"]],"market":[]},{"farmer":["HARVEST"],"hands":[["WEST"],["FERTILIZE"],["NORTH"],["FEED"],["SOUTH"],["SOUTH"],["WEST"],["WEST"],["FERTILIZE"],["WEST"],["HARVEST"]],"market":[]},{"farmer":["SOUTH"],"hands":[["WATER"],["WATER"],["FEED"],["CARE"],["PLACE","STRAWBERRY",4],["WATER"],["WEST"],["FEED"],["WATER"],["WATER"],["EAST"]],"market":[["SELL","FERTILIZER",8],["SELL","STRAWBERRY",4]]},{"farmer":["HARVEST"],"hands":[["SOUTH"],["EAST"],["CARE"],["COLLECT_FERTILIZER"],["WEST"],["SOUTH"],["WEST"],["CARE"],["WEST"],["EAST"],["EAST"]],"market":[]},{"farmer":["EAST"],"hands":[["WATER"],["WATER"],["COLLECT_FERTILIZER"],["NORTH"],["WEST"],["WATER"],["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["NORTH"]],"market":[["SELL","WOOL",6]]},{"farmer":["HARVEST"],"hands":[["HARVEST"],["EAST"],["EAST"],["FEED"],["WEST"],["HARVEST"],["HARVEST"],["SOUTH"],["FERTILIZE"],["SOUTH"],["NORTH"]],"market":[["SELL","MILK",12],["SELL","WHEAT",5],["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[["PLANT","WHEAT"],["WATER"],["FEED"],["CARE"],["WEST"],["PLANT","WHEAT"],["NORTH"],["FEED"],["WATER"],["WATER"],["HARVEST"]],"market":[]},{"farmer":["WEST"],"hands":[["WATER"],["HARVEST"],["CARE"],["COLLECT_FERTILIZER"],["WEST"],["WATER"],["NORTH"],["CARE"],["EAST"],["EAST"],["EAST"]],"market":[["SELL","MILK",6]]},{"farmer":["WEST"],"hands":[["WEST"],["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["WEST"],["NORTH"],["EAST"],["NORTH"],["COLLECT_FERTILIZER"],["WATER"],["WATER"],["NORTH"]],"market":[]},{"farmer":["WEST"],"hands":[["WATER"],["WATER"],["SOUTH"],["SOUTH"],["NORTH"],["WATER"],["WATER"],["WEST"],["HARVEST"],["EAST"],["HARVEST"]],"market":[]},{"farmer":["PLACE","STRAWBERRY",12],"hands":[["SOUTH"],["SOUTH"],["FEED"],["FEED"],["NORTH"],["SOUTH"],["WEST"],["FEED"],["PLANT","WHEAT"],["WATER"],["SOUTH"]],"market":[["SELL","STRAWBERRY",8],["SELL","EGG",6]]},{"farmer":["WEST"],"hands":[["WATER"],["WATER"],["CARE"],["CARE"],["WATER"],["WATER"],["WEST"],["CARE"],["WATER"],["SOUTH"],["SOUTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["SOUTH"],["HARVEST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["HARVEST"],["EAST"],["WEST"],["COLLECT_FERTILIZER"],["WEST"],["WATER"],["SOUTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["WATER"],["PLANT","WHEAT"],["SOUTH"],["WEST"],["PLANT","WHEAT"],["WATER"],["WEST"],["NORTH"],["NORTH"],["WEST"],["PLACE","WOOL",14]],"market":[["SELL","WOOL",3]]},{"farmer":["HARVEST"],"hands":[["SOUTH"],["WATER"],["CARE"],["WATER"],["WATER"],["PASS"],["WATER"],["WATER"],["WATER"],["WATER"],["CARE"]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["WEST"],"hands":[],"market":[["SELL","MILK",21],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",5],["EAST"],["WEST"],["NORTH"],["EAST"],["PICKUP","FERTILIZER",4],["WEST"],["WEST"],["PICKUP","WHEAT",5]],"market":[["SELL","FERTILIZER",8],["SELL","WOOL",12],["HIRE"],["HIRE"],["SELL","FERTILIZER",5]]},{"farmer":["NORTH"],"hands":[["WEST"],["NORTH"],["NORTH"],["PICKUP","WHEAT",2],["NORTH"],["NORTH"],["WEST"],["NORTH"],["EAST"],["NORTH"],["SOUTH"]],"market":[]},{"farmer":["WEST"],"hands":[["FEED"],["NORTH"],["WEST"],["EAST"],["EAST"],["NORTH"],["WEST"],["PICKUP","WHEAT",5],["FEED"],["NORTH"],["SOUTH"]],"market":[]},{"farmer":["WEST"],"hands":[["CARE"],["NORTH"],["WEST"],["FEED"],["HARVEST"],["NORTH"],["WEST"],["FEED"],["CARE"],["WEST"],["SOUTH"]],"market":[]},{"farmer":["HARVEST"],"hands":[["COLLECT_FERTILIZER"],["WATER"],["WEST"],["CARE"],["SOUTH"],["WATER"],["WATER"],["CARE"],["COLLECT_FERTILIZER"],["WEST"],["SOUTH"]],"market":[["SELL","WHEAT",5]]},{"farmer":["NORTH"],"hands":[["NORTH"],["EAST"],["WATER"],["COLLECT_FERTILIZER"],["EAST"],["NORTH"],["SOUTH"],["COLLECT_FERTILIZER"],["NORTH"],["WEST"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["FEED"],["EAST"],["HARVEST"],["HARVEST"],["HARVEST"],["FERTILIZE"],["SOUTH"],["NORTH"],["FEED"],["WEST"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["CARE"],["HARVEST"],["PLANT","WHEAT"],["EAST"],["WEST"],["WATER"],["WATER"],["FEED"],["CARE"],["WATER"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["FEED"],["WEST"],["EAST"],["HARVEST"],["CARE"],["COLLECT_FERTILIZER"],["HARVEST"],["HARVEST"]],"market":[["SELL","STRAWBERRY",10],["BUY_SEED","WHEAT",2]]},{"farmer":["HARVEST"],"hands":[["WEST"],["EAST"],["SOUTH"],["CARE"],["PLACE","WOOL",6],["FERTILIZE"],["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["NORTH"],["PLANT","WHEAT"],["PLANT","WHEAT"]],"market":[]},{"farmer":["NORTH"],"hands":[["FEED"],["HARVEST"],["SOUTH"],["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["WATER"],["NORTH"],["FEED"],["WATER"],["WATER"]],"market":[]},{"farmer":["HARVEST"],"hands":[["CARE"],["SOUTH"],["WATER"],["NORTH"],["NORTH"],["EAST"],["SOUTH"],["FEED"],["CARE"],["NORTH"],["WEST"]],"market":[]},{"farmer":["EAST"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["SOUTH"],["WATER"],["NORTH"],["FERTILIZE"],["SOUTH"],["CARE"],["COLLECT_FERTILIZER"],["EAST"],["WATER"]],"market":[]},{"farmer":["HARVEST"],"hands":[["SOUTH"],["SOUTH"],["EAST"],["NORTH"],["NORTH"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["WATER"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["FEED"],["WEST"],["WATER"],["WATER"],["WEST"],["SOUTH"],["NORTH"],["NORTH"],["EAST"],["HARVEST"],["PLANT","WHEAT"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["CARE"],["WEST"],["SOUTH"],["EAST"],["WATER"],["FERTILIZE"],["NORTH"],["FEED"],["FEED"],["PLANT","WHEAT"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["WEST"],["EAST"],["WATER"],["WEST"],["WATER"],["NORTH"],["CARE"],["CARE"],["WATER"],["NORTH"]],"market":[]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["WEST"],["WATER"],["SOUTH"],["WEST"],["NORTH"],["NORTH"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["WEST"],["PLACE","STRAWBERRY",6],["SOUTH"],["WATER"],["WEST"],["EAST"],["EAST"],["SOUTH"],["SOUTH"],["HARVEST"],["HARVEST"]],"market":[["SELL","STRAWBERRY",6],["BUY_SEED","WHEAT",1],["SELL","EGG",9]]},{"farmer":["EAST"],"hands":[["FEED"],["PASS"],["EAST"],["SOUTH"],["WEST"],["EAST"],["EAST"],["WEST"],["FEED"],["SOUTH"],["PLANT","WHEAT"]],"market":[]},{"farmer":["PLACE","STRAWBERRY",10],"hands":[["CARE"],["PASS"],["WATER"],["WATER"],["WATER"],["WATER"],["EAST"],["FEED"],["CARE"],["EAST"],["WATER"]],"market":[["SELL","STRAWBERRY",12]]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["PASS"],["EAST"],["EAST"],["PASS"],["WEST"],["EAST"],["CARE"],["COLLECT_FERTILIZER"],["EAST"],["PASS"]],"market":[]},{"farmer":["WATER"],"hands":[["PASS"],["PASS"],["WATER"],["WATER"],["PASS"],["WATER"],["DROP"],["COLLECT_FERTILIZER"],["PASS"],["DROP"],["PASS"]],"market":[]},{"farmer":["PICKUP","WHEAT",2],"hands":[],"market":[["SELL","MILK",21],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["EAST"],"hands":[["NORTH"],["PICKUP","FERTILIZER",2],["SOUTH"],["SOUTH"],["PICKUP","WHEAT",3],["PICKUP","WHEAT"],["PICKUP","WHEAT",3],["PICKUP","WHEAT",2],["EAST"]],"market":[["SELL","FERTILIZER",12],["HIRE"],["HIRE"],[],["BUY_PRODUCT","FERTILIZER",1]]},{"farmer":["EAST"],"hands":[["EAST"],["SOUTH"],["WEST"],["WATER"],["EAST"],["WEST"],["WEST"],["EAST"],["EAST"],["PICKUP","WHEAT",3],["PICKUP","WHEAT",2]],"market":[]},{"farmer":["EAST"],"hands":[["NORTH"],["WATER"],["WEST"],["SOUTH"],["FEED"],["WEST"],["WEST"],["HARVEST"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["EAST"],"hands":[["NORTH"],["WEST"],["SOUTH"],["SOUTH"],["CARE"],["WEST"],["WATER"],["WEST"],["WATER"],["FEED"],["NORTH"]],"market":[]},{"farmer":["HARVEST"],"hands":[["NORTH"],["SOUTH"],["SOUTH"],["WATER"],["COLLECT_FERTILIZER"],["NORTH"],["SOUTH"],["FEED"],["HARVEST"],["HARVEST"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["HARVEST"],["WATER"],["WATER"],["SOUTH"],["NORTH"],["FEED"],["WATER"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"],["FEED"]],"market":[]},{"farmer":["HARVEST"],"hands":[["EAST"],["WEST"],["WEST"],["WATER"],["FEED"],["CARE"],["WEST"],["HARVEST"],["HARVEST"],["COLLECT_FERTILIZER"],["CARE"]],"market":[]},{"farmer":["EAST"],"hands":[["EAST"],["WATER"],["WATER"],["SOUTH"],["CARE"],["COLLECT_FERTILIZER"],["WATER"],["WEST"],["WEST"],["NORTH"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["WATER"],["WEST"],["HARVEST"],["WATER"],["COLLECT_FERTILIZER"],["NORTH"],["NORTH"],["FEED"],["NORTH"],["FEED"],["NORTH"]],"market":[["SELL","FERTILIZER",2],["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["HARVEST"],["NORTH"],["WEST"],["WEST"],["NORTH"],["WATER"],["WATER"],["CARE"],["HARVEST"],["CARE"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["PLANT","WHEAT"],["WATER"],["WATER"],["WATER"],["FEED"],["NORTH"],["NORTH"],["COLLECT_FERTILIZER"],["WEST"],["COLLECT_FERTILIZER"],["WEST"]],"market":[]},{"farmer":["WEST"],"hands":[["WATER"],["NORTH"],["NORTH"],["WEST"],["CARE"],["NORTH"],["FEED"],["HARVEST"],["HARVEST"],["HARVEST"],["FEED"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["FEED"],"hands":[["EAST"],["WATER"],["WATER"],["NORTH"],["COLLECT_FERTILIZER"],["FERTILIZE"],["CARE"],["EAST"],["NORTH"],["WEST"],["CARE"]],"market":[["SELL","WHEAT",5],["SELL","FERTILIZER",6]]},{"farmer":["CARE"],"hands":[["WATER"],["WEST"],["HARVEST"],["PLANT","WHEAT"],["HARVEST"],["WATER"],["COLLECT_FERTILIZER"],["EAST"],["HARVEST"],["FEED"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["HARVEST"],["WATER"],["PLANT","WHEAT"],["WATER"],["EAST"],["WEST"],["NORTH"],["DROP"],["SOUTH"],["CARE"],["WEST"]],"market":[[],["BUY_SEED","WHEAT",2]]},{"farmer":["WEST"],"hands":[["PLANT","WHEAT"],["NORTH"],["WATER"],["WEST"],["EAST"],["WATER"],["FEED"],["WEST"],["SOUTH"],["COLLECT_FERTILIZER"],["WATER"]],"market":[]},{"farmer":["WEST"],"hands":[["WATER"],["WATER"],["WEST"],["WEST"],["HARVEST"],["NORTH"],["CARE"],["PASS"],["SOUTH"],["HARVEST"],["WEST"]],"market":[]},{"farmer":["PLACE","STRAWBERRY",6],"hands":[["SOUTH"],["NORTH"],["WATER"],["WATER"],["NORTH"],["WATER"],["COLLECT_FERTILIZER"],["CARE"],["SOUTH"],["WEST"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["WATER"],["FERTILIZE"],["NORTH"],["SOUTH"],["WATER"],["HARVEST"],["EAST"],["NORTH"],["PICKUP","WHEAT"],["WATER"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1],["SELL","EGG",9]]},{"farmer":["FEED"],"hands":[["SOUTH"],["WATER"],["WATER"],["WATER"],["WEST"],["PLANT","WHEAT"],["FEED"],["NORTH"],["PLACE","STRAWBERRY",10],["WEST"],["FERTILIZE"]],"market":[["SELL","STRAWBERRY",4]]},{"farmer":["CARE"],"hands":[["WATER"],["NORTH"],["HARVEST"],["HARVEST"],["HARVEST"],["WATER"],["CARE"],["NORTH"],["FEED"],["PASS"],["WATER"]],"market":[["SELL","STRAWBERRY",12],["SELL","WOOL",24]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["SOUTH"],["FERTILIZE"],["PLANT","WHEAT"],["PLANT","WHEAT"],["NORTH"],["EAST"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"],["PASS"],["EAST"]],"market":[["SELL","WHEAT",2]]},{"farmer":["HARVEST"],"hands":[["WATER"],["WATER"],["WATER"],["WATER"],["HARVEST"],["WATER"],["HARVEST"],["WATER"],["COLLECT_FERTILIZER"],["WATER"],["WATER"]],"market":[["SELL","MILK",14],["SELL","WOOL",12]]},{"farmer":["EAST"],"hands":[],"market":[["SELL","MILK",21],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["PASS"],"hands":[["PICKUP","WHEAT",2],["NORTH"],["NORTH"],["WEST"],["PICKUP","WHEAT",3],["SOUTH"],["NORTH"],["NORTH"],["PASS"]],"market":[[],["HIRE"],[],["SELL","FERTILIZER",4],["SELL","FERTILIZER",3]]},{"farmer":["PICKUP","WHEAT",2],"hands":[["PASS"],["EAST"],["PICKUP","FERTILIZER",3],["WEST"],["FEED"],["SOUTH"],["WEST"],["WEST"],["PICKUP","WHEAT",4],["NORTH"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["NORTH"],"hands":[["FEED"],["PICKUP","FERTILIZER",2],["PICKUP","WHEAT"],["WEST"],["CARE"],["SOUTH"],["PICKUP","WHEAT",3],["WEST"],["EAST"],["PICKUP","WHEAT",2]],"market":[]},{"farmer":["FEED"],"hands":[["CARE"],["EAST"],["EAST"],["WEST"],["COLLECT_FERTILIZER"],["SOUTH"],["NORTH"],["WEST"],["FEED"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["COLLECT_FERTILIZER"],["EAST"],["EAST"],["NORTH"],["NORTH"],["WATER"],["NORTH"],["WEST"],["CARE"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WEST"],["EAST"],["FEED"],["WATER"],["NORTH"],["HARVEST"],["FEED"],["HARVEST"],["COLLECT_FERTILIZER"],["FEED"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["EAST"],"hands":[["FEED"],["EAST"],["CARE"],["NORTH"],["FEED"],["PLANT","WHEAT"],["CARE"],["NORTH"],["HARVEST"],["CARE"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["CARE"],["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["WEST"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["FEED"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["NORTH"],["WATER"],["COLLECT_FERTILIZER"],["WEST"],["WEST"],["EAST"],["PLACE","MILK",3],["HARVEST"]],"market":[["SELL","FERTILIZER",3]]},{"farmer":["CARE"],"hands":[["NORTH"],["SOUTH"],["FERTILIZE"],["HARVEST"],["HARVEST"],["WATER"],["CARE"],["NORTH"],["EAST"],["WEST"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["NORTH"],["FERTILIZE"],["WATER"],["PLANT","WHEAT"],["WEST"],["HARVEST"],["FEED"],["HARVEST"],["NORTH"],["HARVEST"]],"market":[["SELL","WHEAT",2]]},{"farmer":["NORTH"],"hands":[["NORTH"],["WATER"],["NORTH"],["WATER"],["NORTH"],["PLANT","WHEAT"],["HARVEST"],["NORTH"],["FEED"],["FEED"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["FERTILIZE"],"hands":[["WATER"],["WEST"],["FERTILIZE"],["EAST"],["FEED"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["CARE"],["CARE"]],"market":[]},{"farmer":["WATER"],"hands":[["EAST"],["WEST"],["WATER"],["EAST"],["CARE"],["WEST"],["WEST"],["EAST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"]],"market":[["SELL","MILK",6]]},{"farmer":["EAST"],"hands":[["EAST"],["WEST"],["EAST"],["WATER"],["COLLECT_FERTILIZER"],["WATER"],["WATER"],["EAST"],["WEST"],["WEST"]],"market":[[]]},{"farmer":["WATER"],"hands":[["FERTILIZE"],["WEST"],["FERTILIZE"],["NORTH"],["HARVEST"],["WEST"],["SOUTH"],["EAST"],["WEST"],["WATER"]],"market":[]},{"farmer":["EAST"],"hands":[["WATER"],["PLACE","STRAWBERRY",2],["WATER"],["HARVEST"],["NORTH"],["WATER"],["FEED"],["SOUTH"],["FEED"],["SOUTH"]],"market":[["SELL","WOOL",3]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["EAST"],["EAST"],["PASS"],["WATER"],["NORTH"],["CARE"],["SOUTH"],["CARE"],["WATER"]],"market":[["SELL","STRAWBERRY",2],["SELL","STRAWBERRY",1],["SELL","STRAWBERRY",5]]},{"farmer":["EAST"],"hands":[["WATER"],["EAST"],["HARVEST"],["PASS"],["HARVEST"],["WATER"],["COLLECT_FERTILIZER"],["SOUTH"],["COLLECT_FERTILIZER"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["EAST"],["EAST"],["WEST"],["PASS"],["PLANT","WHEAT"],["WEST"],["WEST"],["SOUTH"],["WEST"],["PLANT","WHEAT"]],"market":[]},{"farmer":["HARVEST"],"hands":[["WATER"],["FERTILIZE"],["SOUTH"],["PASS"],["WATER"],["WATER"],["WATER"],["DROP"],["FEED"],["WATER"]],"market":[["SELL","STRAWBERRY",8]]},{"farmer":["PLANT","WHEAT"],"hands":[["EAST"],["WATER"],["FERTILIZE"],["PASS"],["WEST"],["NORTH"],["NORTH"],["PASS"],["CARE"],["SOUTH"]],"market":[["SELL","WOOL",6]]},{"farmer":["WATER"],"hands":[["WATER"],["PASS"],["WATER"],["PASS"],["WATER"],["WATER"],["WATER"],["PASS"],["COLLECT_FERTILIZER"],["WATER"]],"market":[]},{"farmer":["PICKUP","WHEAT"],"hands":[],"market":[["SELL","MILK",3],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["SOUTH"],"hands":[["NORTH"],["PICKUP","FERTILIZER",6],["PICKUP","FERTILIZER",4],["PICKUP","WHEAT",3],["NORTH"],["PASS"],["PICKUP","WHEAT",2],["PICKUP","WHEAT",2],["PICKUP","WHEAT"]],"market":[["HIRE"],["HIRE"],[]]},{"farmer":["PICKUP","FERTILIZER"],"hands":[["EAST"],["SOUTH"],["WEST"],["FEED"],["NORTH"],["PICKUP","FERTILIZER"],["EAST"],["NORTH"],["EAST"],["PICKUP","WHEAT",4],["PICKUP","WHEAT",2]],"market":[]},{"farmer":["WEST"],"hands":[["EAST"],["SOUTH"],["FERTILIZE"],["COLLECT_FERTILIZER"],["NORTH"],["SOUTH"],["EAST"],["NORTH"],["FEED"],["NORTH"],["WEST"]],"market":[]},{"farmer":["WEST"],"hands":[["HARVEST"],["SOUTH"],["WATER"],["CARE"],["HARVEST"],["FERTILIZE"],["NORTH"],["FEED"],["CARE"],["PASS"],["WEST"]],"market":[]},{"farmer":["FERTILIZE"],"hands":[["EAST"],["FERTILIZE"],["WEST"],["HARVEST"],["NORTH"],["WATER"],["NORTH"],["CARE"],["COLLECT_FERTILIZER"],["FEED"],["FEED"]],"market":[["SELL","FERTILIZER",3],["SELL","WHEAT",4]]},{"farmer":["WATER"],"hands":[["HARVEST"],["WATER"],["FERTILIZE"],["PLACE","MILK",3],["HARVEST"],["WEST"],["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"],["CARE"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["SOUTH"],["NORTH"],["WATER"],["NORTH"],["EAST"],["SOUTH"],["HARVEST"],["HARVEST"],["NORTH"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["NORTH"],"hands":[["HARVEST"],["FERTILIZE"],["SOUTH"],["FEED"],["SOUTH"],["SOUTH"],["EAST"],["WEST"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["FEED"],"hands":[["EAST"],["WATER"],["FERTILIZE"],["HARVEST"],["HARVEST"],["WATER"],["HARVEST"],["FEED"],["NORTH"],["FEED"],["HARVEST"]],"market":[["SELL","FERTILIZER",6]]},{"farmer":["CARE"],"hands":[["HARVEST"],["WEST"],["WATER"],["CARE"],["WEST"],["WEST"],["WEST"],["CARE"],["HARVEST"],["CARE"],["FEED"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["NORTH"],["FERTILIZE"],["WEST"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["WEST"],["COLLECT_FERTILIZER"],["EAST"],["COLLECT_FERTILIZER"],["CARE"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["NORTH"],"hands":[["WATER"],["WATER"],["FERTILIZE"],["NORTH"],["SOUTH"],["WEST"],["FEED"],["NORTH"],["HARVEST"],["EAST"],["COLLECT_FERTILIZER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WEST"],["WEST"],["WATER"],["NORTH"],["SOUTH"],["WATER"],["CARE"],["WATER"],["SOUTH"],["SOUTH"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["WEST"],["FERTILIZE"],["WEST"],["CARE"],["PICKUP","WHEAT"],["SOUTH"],["COLLECT_FERTILIZER"],["HARVEST"],["HARVEST"],["FEED"],["WATER"]],"market":[["SELL","STRAWBERRY",8]]},{"farmer":["WATER"],"hands":[["WEST"],["WATER"],["WEST"],["HARVEST"],["PLACE","STRAWBERRY",6],["WATER"],["WEST"],["PLANT","WHEAT"],["EAST"],["CARE"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["WEST"],["WEST"],["WATER"],["FEED"],["PASS"],["HARVEST"],["WEST"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["PLANT","WHEAT"]],"market":[]},{"farmer":["WEST"],"hands":[["SOUTH"],["WATER"],["SOUTH"],["COLLECT_FERTILIZER"],["EAST"],["PLANT","WHEAT"],["SOUTH"],["WEST"],["NORTH"],["HARVEST"],["WATER"]],"market":[]},{"farmer":["PASS"],"hands":[["DROP"],["NORTH"],["WATER"],["WEST"],["EAST"],["WATER"],["SOUTH"],["WATER"],["WATER"],["WEST"],["NORTH"]],"market":[["SELL","STRAWBERRY",6],["SELL","STRAWBERRY",4],["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["PICKUP","WHEAT"],["FERTILIZE"],["SOUTH"],["NORTH"],["FEED"],["WEST"],["PLACE","STRAWBERRY",4],["HARVEST"],["EAST"],["WEST"],["WATER"]],"market":[["SELL","EGG",12]]},{"farmer":["SOUTH"],"hands":[["FEED"],["WATER"],["WATER"],["WATER"],["CARE"],["WATER"],["WEST"],["PLANT","WHEAT"],["WATER"],["WEST"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["CARE"],["NORTH"],["HARVEST"],["HARVEST"],["COLLECT_FERTILIZER"],["EAST"],["FEED"],["WATER"],["SOUTH"],["FEED"],["WATER"]],"market":[["SELL","MILK",12],["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["FERTILIZE"],["PLANT","WHEAT"],["PLANT","WHEAT"],["HARVEST"],["EAST"],["CARE"],["NORTH"],["SOUTH"],["CARE"],["WEST"]],"market":[["SELL","STRAWBERRY",8]]},{"farmer":["WATER"],"hands":[["HARVEST"],["WATER"],["WATER"],["WATER"],["PASS"],["WATER"],["COLLECT_FERTILIZER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["WATER"]],"market":[["SELL","MILK",6],["SELL","WOOL",24]]},{"farmer":["PICKUP","WHEAT",5],"hands":[],"market":[["SELL","FERTILIZER",14],["SELL","MILK",4],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["EAST"],"hands":[["PICKUP","WHEAT",2],["WEST"],["WEST"],["PICKUP","WHEAT",5],["PICKUP","FERTILIZER",3],["WEST"],["WEST"],["PICKUP","WHEAT",5]],"market":[["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["FEED"],"hands":[["EAST"],["WEST"],["SOUTH"],["WEST"],["NORTH"],["SOUTH"],["NORTH"],["FEED"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["CARE"],"hands":[["FEED"],["HARVEST"],["SOUTH"],["FEED"],["NORTH"],["SOUTH"],["WEST"],["CARE"],["NORTH"],["NORTH"],["NORTH"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["WEST"],["SOUTH"],["CARE"],["NORTH"],["HARVEST"],["WEST"],["COLLECT_FERTILIZER"],["NORTH"],["NORTH"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["SOUTH"],["COLLECT_FERTILIZER"],["NORTH"],["EAST"],["WEST"],["NORTH"],["NORTH"],["EAST"],["WEST"]],"market":[["SELL","WHEAT",5]]},{"farmer":["NORTH"],"hands":[["HARVEST"],["SOUTH"],["WATER"],["NORTH"],["FERTILIZE"],["HARVEST"],["WEST"],["FEED"],["WATER"],["EAST"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["EAST"],["HARVEST"],["WEST"],["FEED"],["WATER"],["SOUTH"],["WATER"],["CARE"],["NORTH"],["WATER"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["CARE"],"hands":[["FEED"],["EAST"],["WATER"],["CARE"],["EAST"],["HARVEST"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"],["NORTH"],["PLANT","WHEAT"]],"market":[["SELL","WHEAT",4],["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["HARVEST"],["WEST"],["COLLECT_FERTILIZER"],["FERTILIZE"],["NORTH"],["PLANT","WHEAT"],["NORTH"],["WATER"],["NORTH"],["WATER"]],"market":[["SELL","STRAWBERRY",13]]},{"farmer":["NORTH"],"hands":[["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["WEST"],["WATER"],["NORTH"],["WATER"],["FEED"],["WEST"],["WATER"],["SOUTH"]],"market":[]},{"farmer":["FEED"],"hands":[["NORTH"],["HARVEST"],["HARVEST"],["FEED"],["SOUTH"],["HARVEST"],["SOUTH"],["CARE"],["WEST"],["SOUTH"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["CARE"],"hands":[["WATER"],["NORTH"],["PLANT","WHEAT"],["CARE"],["EAST"],["NORTH"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["WEST"],["WATER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["EAST"],["EAST"],["WATER"],["COLLECT_FERTILIZER"],["FERTILIZE"],["PLACE","STRAWBERRY",8],["SOUTH"],["NORTH"],["DIG"],["WATER"],["HARVEST"]],"market":[["SELL","STRAWBERRY",8],["BUY_SEED","WHEAT",2]]},{"farmer":["HARVEST"],"hands":[["WATER"],["HARVEST"],["NORTH"],["SOUTH"],["WATER"],["NORTH"],["SOUTH"],["FEED"],["PLANT","WHEAT"],["WEST"],["PLANT","WHEAT"]],"market":[["SELL","STRAWBERRY",2]]},{"farmer":["EAST"],"hands":[["SOUTH"],["NORTH"],["WEST"],["FEED"],["EAST"],["NORTH"],["WATER"],["CARE"],["WATER"],["WEST"],["WATER"]],"market":[]},{"farmer":["FEED"],"hands":[["WATER"],["HARVEST"],["WATER"],["CARE"],["EAST"],["WEST"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"],["WEST"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["CARE"],"hands":[["EAST"],["EAST"],["HARVEST"],["COLLECT_FERTILIZER"],["WATER"],["WEST"],["PLANT","WHEAT"],["SOUTH"],["HARVEST"],["WEST"],["WEST"]],"market":[["SELL","WOOL",24],["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["WATER"],["HARVEST"],["PLANT","WHEAT"],["HARVEST"],["NORTH"],["WEST"],["WATER"],["WEST"],["DIG"],["WEST"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["NORTH"],["PLACE","STRAWBERRY",16],["WATER"],["WEST"],["WATER"],["WEST"],["EAST"],["FEED"],["PLANT","WHEAT"],["HARVEST"],["DIG"]],"market":[["SELL","STRAWBERRY",16],["BUY_SEED","WHEAT",1],["SELL","EGG",9]]},{"farmer":["FEED"],"hands":[["HARVEST"],["NORTH"],["EAST"],["FEED"],["WEST"],["HARVEST"],["WATER"],["CARE"],["WATER"],["DIG"],["PLANT","WHEAT"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["CARE"],"hands":[["NORTH"],["NORTH"],["WATER"],["CARE"],["WATER"],["DIG"],["NORTH"],["COLLECT_FERTILIZER"],["WEST"],["PLANT","WHEAT"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["HARVEST"],["EAST"],["EAST"],["COLLECT_FERTILIZER"],["SOUTH"],["PLANT","WHEAT"],["WEST"],["SOUTH"],["WATER"],["WATER"],["NORTH"]],"market":[]},{"farmer":["PASS"],"hands":[["PASS"],["HARVEST"],["HARVEST"],["PASS"],["HARVEST"],["WATER"],["WATER"],["HARVEST"],["PASS"],["PASS"],["WATER"]],"market":[["SELL","MILK",21]]},{"farmer":["PICKUP","WHEAT",2],"hands":[],"market":[["SELL","STRAWBERRY",13],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["NORTH"],"hands":[["NORTH"],["NORTH"],["WEST"],["PICKUP","WHEAT",4],["PICKUP","WHEAT",3],["WATER"],["NORTH"],["WEST"],["PICKUP","WHEAT",2]],"market":[["SELL","MILK",14],["SELL","MILK",3],["HIRE"],[],["BUY_PRODUCT","FERTILIZER",4]]},{"farmer":["HARVEST"],"hands":[["EAST"],["PICKUP","WHEAT",3],["PICKUP","FERTILIZER"],["WEST"],["EAST"],["PICKUP","WHEAT",2],["PICKUP","WHEAT"],["WEST"],["NORTH"],["WEST"]],"market":[]},{"farmer":["SOUTH"],"hands":[["NORTH"],["NORTH"],["SOUTH"],["FEED"],["EAST"],["WEST"],["EAST"],["NORTH"],["NORTH"],["SOUTH"]],"market":[]},{"farmer":["PLACE","MILK",3],"hands":[["NORTH"],["NORTH"],["SOUTH"],["CARE"],["EAST"],["WATER"],["EAST"],["NORTH"],["NORTH"],["WATER"]],"market":[["SELL","FERTILIZER",10],["SELL","FERTILIZER",1]]},{"farmer":["FEED"],"hands":[["HARVEST"],["FEED"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["WEST"],["FEED"],["WATER"],["HARVEST"],["WEST"]],"market":[]},{"farmer":["CARE"],"hands":[["EAST"],["CARE"],["SOUTH"],["HARVEST"],["WEST"],["WATER"],["CARE"],["WEST"],["NORTH"],["WATER"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["HARVEST"],["COLLECT_FERTILIZER"],["WATER"],["EAST"],["NORTH"],["SOUTH"],["COLLECT_FERTILIZER"],["WATER"],["HARVEST"],["SOUTH"]],"market":[[]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["HARVEST"],["WEST"],["PLACE","WOOL",4],["HARVEST"],["WATER"],["EAST"],["EAST"],["WEST"],["WATER"]],"market":[]},{"farmer":["WEST"],"hands":[["HARVEST"],["NORTH"],["FERTILIZE"],["WEST"],["WEST"],["WEST"],["NORTH"],["NORTH"],["WATER"],["WEST"]],"market":[[]]},{"farmer":["WEST"],"hands":[["EAST"],["FEED"],["WATER"],["NORTH"],["FEED"],["WATER"],["HARVEST"],["WATER"],["SOUTH"],["WATER"]],"market":[["SELL","FERTILIZER",9]]},{"farmer":["FEED"],"hands":[["WATER"],["CARE"],["WEST"],["FEED"],["CARE"],["NORTH"],["EAST"],["EAST"],["SOUTH"],["WEST"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["CARE"],"hands":[["HARVEST"],["COLLECT_FERTILIZER"],["WATER"],["CARE"],["COLLECT_FERTILIZER"],["WATER"],["SOUTH"],["WATER"],["SOUTH"],["WATER"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["PLANT","WHEAT"],["HARVEST"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"],["NORTH"],["HARVEST"],["NORTH"],["SOUTH"],["HARVEST"]],"market":[]},{"farmer":["WEST"],"hands":[["WATER"],["EAST"],["PLANT","WHEAT"],["EAST"],["SOUTH"],["HARVEST"],["NORTH"],["WATER"],["PLACE","STRAWBERRY",4],["PLANT","WHEAT"]],"market":[["SELL","WHEAT",2]]},{"farmer":["WEST"],"hands":[["SOUTH"],["EAST"],["WATER"],["FEED"],["PLACE","STRAWBERRY",4],["FEED"],["WATER"],["WEST"],["EAST"],["WATER"]],"market":[["SELL","STRAWBERRY",5]]},{"farmer":["WATER"],"hands":[["WATER"],["NORTH"],["WEST"],["CARE"],["NORTH"],["CARE"],["NORTH"],["WEST"],["FEED"],["WEST"]],"market":[]},{"farmer":["SOUTH"],"hands":[["EAST"],["HARVEST"],["SOUTH"],["COLLECT_FERTILIZER"],["FEED"],["COLLECT_FERTILIZER"],["WATER"],["WEST"],["CARE"],["SOUTH"]],"market":[["SELL","STRAWBERRY",3],["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WATER"],["SOUTH"],["WATER"],["WEST"],["CARE"],["NORTH"],["WEST"],["WATER"],["COLLECT_FERTILIZER"],["WATER"]],"market":[["BUY_SEED","WHEAT",3]]},{"farmer":["SOUTH"],"hands":[["NORTH"],["SOUTH"],["EAST"],["NORTH"],["COLLECT_FERTILIZER"],["WATER"],["DIG"],["HARVEST"],["HARVEST"],["SOUTH"]],"market":[["SELL","EGG",6]]},{"farmer":["WATER"],"hands":[["WATER"],["FEED"],["EAST"],["FEED"],["NORTH"],["EAST"],["PLANT","WHEAT"],["PLANT","WHEAT"],["EAST"],["WATER"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["HARVEST"],["CARE"],["WATER"],["CARE"],["FEED"],["FEED"],["WATER"],["WATER"],["FEED"],["HARVEST"]],"market":[["SELL","MILK",18],[],["SELL","WOOL",12]]},{"farmer":["PLANT","WHEAT"],"hands":[["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["EAST"],["COLLECT_FERTILIZER"],["CARE"],["CARE"],["WEST"],["SOUTH"],["CARE"],["PLANT","WHEAT"]],"market":[[],["SELL","WOOL",12]]},{"farmer":["WATER"],"hands":[["WATER"],["HARVEST"],["WATER"],["HARVEST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["HARVEST"],["WATER"],["COLLECT_FERTILIZER"],["WATER"]],"market":[[]]},{"farmer":["WEST"],"hands":[],"market":[["SELL","FERTILIZER",24],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",3],["PICKUP","WHEAT",2],["EAST"],["PICKUP","WHEAT",3],["PICKUP","WHEAT",4],["PICKUP","WHEAT"],["HARVEST"]],"market":[["HIRE"],["HIRE"],["HIRE"],[],["BUY_PRODUCT","FERTILIZER",4]]},{"farmer":["WEST"],"hands":[["WEST"],["EAST"],["NORTH"],["WEST"],["FEED"],["NORTH"],["SOUTH"],["NORTH"],["WEST"],["WEST"]],"market":[["SELL","FERTILIZER",4]]},{"farmer":["SOUTH"],"hands":[["FEED"],["NORTH"],["PICKUP","WHEAT"],["WEST"],["CARE"],["NORTH"],["HARVEST"],["PICKUP","WHEAT"],["WEST"],["PICKUP","WHEAT",2]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["SOUTH"],"hands":[["CARE"],["FEED"],["EAST"],["HARVEST"],["COLLECT_FERTILIZER"],["FEED"],["NORTH"],["WEST"],["WEST"],["NORTH"]],"market":[]},{"farmer":["HARVEST"],"hands":[["COLLECT_FERTILIZER"],["CARE"],["EAST"],["WEST"],["NORTH"],["CARE"],["DROP"],["NORTH"],["NORTH"],["NORTH"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["EAST"],"hands":[["WEST"],["COLLECT_FERTILIZER"],["FEED"],["HARVEST"],["FEED"],["COLLECT_FERTILIZER"],["SOUTH"],["NORTH"],["WATER"],["FEED"]],"market":[["SELL","FERTILIZER",1],["BUY_SEED","WHEAT",2]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["HARVEST"],["CARE"],["NORTH"],["CARE"],["HARVEST"],["SOUTH"],["FEED"],["WEST"],["CARE"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["EAST"],["COLLECT_FERTILIZER"],["FEED"],["COLLECT_FERTILIZER"],["NORTH"],["HARVEST"],["CARE"],["WATER"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["HARVEST"],"hands":[["FEED"],["DIG"],["HARVEST"],["CARE"],["EAST"],["DIG"],["SOUTH"],["COLLECT_FERTILIZER"],["NORTH"],["NORTH"]],"market":[["SELL","STRAWBERRY",4],["BUY_SEED","WHEAT",1]]},{"farmer":["WEST"],"hands":[["CARE"],["PLANT","WHEAT"],["EAST"],["COLLECT_FERTILIZER"],["FEED"],["PLANT","WHEAT"],["HARVEST"],["HARVEST"],["WATER"],["FEED"]],"market":[["SELL","STRAWBERRY",4]]},{"farmer":["WEST"],"hands":[["COLLECT_FERTILIZER"],["WATER"],["DIG"],["HARVEST"],["CARE"],["WATER"],["SOUTH"],["WEST"],["NORTH"],["CARE"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WEST"],["NORTH"],["PLANT","WHEAT"],["EAST"],["COLLECT_FERTILIZER"],["EAST"],["WATER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["SOUTH"],["DIG"],["WATER"],["EAST"],["WEST"],["EAST"],["HARVEST"],["NORTH"],["HARVEST"],["NORTH"]],"market":[["SELL","MILK",6],["BUY_SEED","WHEAT",2]]},{"farmer":["WATER"],"hands":[["FEED"],["PLANT","WHEAT"],["NORTH"],["PLACE","STRAWBERRY",4],["WEST"],["WATER"],["PLANT","WHEAT"],["WATER"],["PLANT","WHEAT"],["WATER"]],"market":[["SELL","WHEAT",2],["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["CARE"],["WATER"],["DIG"],["EAST"],["FEED"],["EAST"],["WATER"],["NORTH"],["WATER"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["PLANT","WHEAT"],"hands":[["COLLECT_FERTILIZER"],["WEST"],["PLANT","WHEAT"],["FEED"],["CARE"],["HARVEST"],["WEST"],["WATER"],["EAST"],["PLANT","WHEAT"]],"market":[]},{"farmer":["WATER"],"hands":[["SOUTH"],["FEED"],["WATER"],["CARE"],["COLLECT_FERTILIZER"],["EAST"],["PASS"],["WEST"],["WATER"],["WATER"]],"market":[["SELL","STRAWBERRY",8],["SELL","STRAWBERRY",5]]},{"farmer":["EAST"],"hands":[["HARVEST"],["CARE"],["EAST"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["WATER"],["WATER"],["SOUTH"],["EAST"]],"market":[["SELL","MILK",14]]},{"farmer":["WATER"],"hands":[["WEST"],["COLLECT_FERTILIZER"],["HARVEST"],["EAST"],["SOUTH"],["HARVEST"],["HARVEST"],["EAST"],["WATER"],["WATER"]],"market":[[],[],["BUY_SEED","WHEAT",2],["SELL","EGG",9]]},{"farmer":["SOUTH"],"hands":[["WATER"],["NORTH"],["SOUTH"],["FEED"],["SOUTH"],["PLANT","WHEAT"],["PLANT","WHEAT"],["EAST"],["WEST"],["EAST"]],"market":[]},{"farmer":["WATER"],"hands":[["HARVEST"],["DIG"],["DIG"],["CARE"],["HARVEST"],["WATER"],["WATER"],["WATER"],["SOUTH"],["WATER"]],"market":[["SELL","WOOL",24],[],["SELL","WOOL",3]]},{"farmer":["EAST"],"hands":[["PLANT","WHEAT"],["PLANT","WHEAT"],["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["SOUTH"],["SOUTH"],["NORTH"],["SOUTH"],["SOUTH"],["EAST"]],"market":[["SELL","WHEAT",7],[]]},{"farmer":["WATER"],"hands":[["WATER"],["WATER"],["WATER"],["HARVEST"],["HARVEST"],["HARVEST"],["HARVEST"],["WATER"],["WATER"],["WATER"]],"market":[[]]},{"farmer":["WEST"],"hands":[],"market":[["SELL","WOOL",3],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["PASS"],"hands":[["PASS"],["NORTH"],["PICKUP","FERTILIZER",3],["WEST"],["SOUTH"],["PASS"],["PICKUP","FERTILIZER"],["NORTH"],["PICKUP","WHEAT",2]],"market":[["HIRE"],[],["BUY_PRODUCT","FERTILIZER",4]]},{"farmer":["NORTH"],"hands":[["PICKUP","WHEAT",2],["WEST"],["SOUTH"],["PICKUP","FERTILIZER",3],["PICKUP","FERTILIZER",4],["PICKUP","WHEAT",2],["FERTILIZE"],["PICKUP","WHEAT",2],["FEED"],["NORTH"]],"market":[]},{"farmer":["NORTH"],"hands":[["WEST"],["SOUTH"],["FERTILIZE"],["WEST"],["WEST"],["FEED"],["PICKUP","WHEAT",2],["NORTH"],["CARE"],["PICKUP","WHEAT"]],"market":[["SELL","FERTILIZER",9]]},{"farmer":["NORTH"],"hands":[["WEST"],["PICKUP","WHEAT",2],["WATER"],["FERTILIZE"],["SOUTH"],["CARE"],["WATER"],["HARVEST"],["COLLECT_FERTILIZER"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["FEED"],["WEST"],["SOUTH"],["WATER"],["FERTILIZE"],["COLLECT_FERTILIZER"],["NORTH"],["CARE"],["HARVEST"],["NORTH"]],"market":[]},{"farmer":["HARVEST"],"hands":[["CARE"],["FEED"],["FERTILIZE"],["WEST"],["WATER"],["HARVEST"],["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["PASS"]],"market":[["SELL","WOOL",12]]},{"farmer":["WEST"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["WATER"],["SOUTH"],["SOUTH"],["EAST"],["NORTH"],["EAST"],["HARVEST"],["FEED"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["WATER"],"hands":[["WEST"],["CARE"],["SOUTH"],["FERTILIZE"],["FERTILIZE"],["FEED"],["FEED"],["FEED"],["FEED"],["CARE"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["FEED"],["COLLECT_FERTILIZER"],["FERTILIZE"],["WATER"],["WATER"],["CARE"],["CARE"],["CARE"],["CARE"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["PLANT","WHEAT"],"hands":[["CARE"],["NORTH"],["WATER"],["WEST"],["WEST"],["COLLECT_FERTILIZER"],["HARVEST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["NORTH"]],"market":[["SELL","WHEAT",6],["BUY_SEED","WHEAT",2]]},{"farmer":["WATER"],"hands":[["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["WEST"],["FERTILIZE"],["WATER"],["EAST"],["COLLECT_FERTILIZER"],["NORTH"],["WEST"],["WATER"]],"market":[]},{"farmer":["NORTH"],"hands":[["NORTH"],["CARE"],["WATER"],["WATER"],["NORTH"],["COLLECT_FERTILIZER"],["NORTH"],["FEED"],["WEST"],["NORTH"]],"market":[]},{"farmer":["WATER"],"hands":[["WATER"],["HARVEST"],["WEST"],["WEST"],["NORTH"],["CARE"],["FEED"],["CARE"],["HARVEST"],["HARVEST"]],"market":[["BUY_SEED","WHEAT",2]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["EAST"],["WATER"],["WATER"],["FERTILIZE"],["EAST"],["CARE"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["EAST"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["PLANT","CARROT"],"hands":[["WATER"],["SOUTH"],["SOUTH"],["SOUTH"],["WATER"],["EAST"],["COLLECT_FERTILIZER"],["HARVEST"],["CARE"],["HARVEST"]],"market":[["SELL","WHEAT",2],["BUY_SEED","CARROT",1]]},{"farmer":["WATER"],"hands":[["HARVEST"],["PLACE","WOOL",8],["WATER"],["WATER"],["WEST"],["NORTH"],["WEST"],["EAST"],["NORTH"],["EAST"]],"market":[]},{"farmer":["WEST"],"hands":[["PLANT","WHEAT"],["WEST"],["WEST"],["HARVEST"],["FERTILIZE"],["DIG"],["PLANT","WHEAT"],["EAST"],["WATER"],["HARVEST"]],"market":[]},{"farmer":["WATER"],"hands":[["WATER"],["NORTH"],["WATER"],["PLANT","WHEAT"],["WATER"],["PLANT","WHEAT"],["WATER"],["WATER"],["WEST"],["SOUTH"]],"market":[["BUY_SEED","CARROT",1],["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["NORTH"],["HARVEST"],["WATER"],["WEST"],["WATER"],["NORTH"],["NORTH"],["WEST"],["HARVEST"]],"market":[["BUY_SEED","CARROT",1],["BUY_SEED","WHEAT",1],["SELL","EGG",3]]},{"farmer":["PLANT","WHEAT"],"hands":[["WATER"],["FEED"],["PLANT","CARROT"],["EAST"],["NORTH"],["NORTH"],["WATER"],["NORTH"],["WATER"],["EAST"]],"market":[["BUY_SEED","WHEAT",1],["SELL","FERTILIZER",1]]},{"farmer":["WATER"],"hands":[["HARVEST"],["CARE"],["WATER"],["WATER"],["WATER"],["DIG"],["HARVEST"],["WATER"],["HARVEST"],["DIG"]],"market":[["SELL","STRAWBERRY",10],["SELL","WOOL",12]]},{"farmer":["WEST"],"hands":[["PLANT","WHEAT"],["COLLECT_FERTILIZER"],["WEST"],["SOUTH"],["NORTH"],["PLANT","WHEAT"],["PLANT","WHEAT"],["EAST"],["PLANT","CARROT"],["PLANT","CARROT"]],"market":[["SELL","WHEAT",2],["SELL","MILK",3]]},{"farmer":["WATER"],"hands":[["WATER"],["HARVEST"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"]],"market":[["SELL","STRAWBERRY",11],["SELL","MILK",3],["SELL","MILK",3],["SELL","MILK",14]]},{"farmer":["PICKUP","WHEAT"],"hands":[],"market":[["SELL","FERTILIZER",11],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["PICKUP","WHEAT",2],["HARVEST"],["NORTH"],["PICKUP","WHEAT",3],["SOUTH"],["PICKUP","FERTILIZER"]],"market":[["SELL","FERTILIZER",2],["HIRE"],["HIRE"],["HIRE"],["HIRE"],[],["BUY_PRODUCT","FERTILIZER",2]]},{"farmer":["WEST"],"hands":[["EAST"],["WEST"],["PICKUP","WHEAT",2],["WEST"],["NORTH"],["WEST"],["SOUTH"],["PICKUP","WHEAT",2],["WEST"],["WEST"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["FEED"],"hands":[["FEED"],["HARVEST"],["EAST"],["FEED"],["NORTH"],["SOUTH"],["SOUTH"],["NORTH"],["WEST"],["PICKUP","WHEAT"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["CARE"],"hands":[["CARE"],["EAST"],["NORTH"],["CARE"],["NORTH"],["HARVEST"],["HARVEST"],["FEED"],["WEST"],["WEST"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["COLLECT_FERTILIZER"],["DROP"],["PASS"],["COLLECT_FERTILIZER"],["NORTH"],["SOUTH"],["SOUTH"],["CARE"],["HARVEST"],["NORTH"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["HARVEST"],"hands":[["HARVEST"],["NORTH"],["PASS"],["NORTH"],["WATER"],["HARVEST"],["HARVEST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["NORTH"]],"market":[]},{"farmer":["WEST"],"hands":[["EAST"],["PICKUP","WHEAT",4],["PASS"],["FEED"],["NORTH"],["SOUTH"],["SOUTH"],["NORTH"],["CARE"],["HARVEST"]],"market":[]},{"farmer":["SOUTH"],"hands":[["FEED"],["FEED"],["FEED"],["CARE"],["PASS"],["HARVEST"],["HARVEST"],["FEED"],["WEST"],["COLLECT_FERTILIZER"]],"market":[["SELL","FERTILIZER",1],["BUY_SEED","CARROT",2]]},{"farmer":["HARVEST"],"hands":[["CARE"],["COLLECT_FERTILIZER"],["CARE"],["COLLECT_FERTILIZER"],["DIG"],["FERTILIZE"],["SOUTH"],["CARE"],["WATER"],["CARE"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["EAST"],"hands":[["COLLECT_FERTILIZER"],["CARE"],["COLLECT_FERTILIZER"],["WEST"],["PLANT","CARROT"],["WEST"],["WATER"],["COLLECT_FERTILIZER"],["HARVEST"],["NORTH"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["NORTH"],["NORTH"],["FEED"],["WATER"],["WATER"],["WEST"],["EAST"],["NORTH"],["WATER"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["SOUTH"],"hands":[["WATER"],["FEED"],["FEED"],["CARE"],["EAST"],["WEST"],["WATER"],["NORTH"],["WATER"],["EAST"]],"market":[["SELL","MILK",3],["BUY_SEED","CARROT",1]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["CARE"],["CARE"],["NORTH"],["DIG"],["NORTH"],["WEST"],["NORTH"],["HARVEST"],["HARVEST"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["SOUTH"],"hands":[["WATER"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["WATER"],["PLANT","CARROT"],["WATER"],["WATER"],["WATER"],["PLANT","CARROT"],["PASS"]],"market":[["SELL","WHEAT",4]]},{"farmer":["HARVEST"],"hands":[["EAST"],["NORTH"],["EAST"],["HARVEST"],["WATER"],["NORTH"],["HARVEST"],["EAST"],["WATER"],["PASS"]],"market":[["SELL","STRAWBERRY",1]]},{"farmer":["WEST"],"hands":[["EAST"],["FEED"],["EAST"],["PLANT","CARROT"],["EAST"],["HARVEST"],["PLANT","CARROT"],["DIG"],["NORTH"],["PASS"]],"market":[["SELL","WHEAT",8],["BUY_SEED","CARROT",1]]},{"farmer":["SOUTH"],"hands":[["NORTH"],["CARE"],["WATER"],["WATER"],["EAST"],["WEST"],["WATER"],["PLANT","CARROT"],["NORTH"],["PASS"]],"market":[["SELL","WOOL",24],["SELL","STRAWBERRY",1],["BUY_SEED","CARROT",1]]},{"farmer":["WATER"],"hands":[["WATER"],["COLLECT_FERTILIZER"],["SOUTH"],["WEST"],["WATER"],["WATER"],["WEST"],["WATER"],["WATER"],["PASS"]],"market":[["SELL","WHEAT",5]]},{"farmer":["HARVEST"],"hands":[["NORTH"],["EAST"],["WATER"],["SOUTH"],["HARVEST"],["NORTH"],["WEST"],["NORTH"],["NORTH"],["FEED"]],"market":[["SELL","STRAWBERRY",1]]},{"farmer":["PLANT","CARROT"],"hands":[["WATER"],["FEED"],["SOUTH"],["WATER"],["PLANT","CARROT"],["WATER"],["WATER"],["DIG"],["WATER"],["CARE"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["WATER"],"hands":[["HARVEST"],["CARE"],["WATER"],["HARVEST"],["PASS"],["NORTH"],["HARVEST"],["PLANT","CARROT"],["HARVEST"],["COLLECT_FERTILIZER"]],"market":[["SELL","MILK",12],["BUY_SEED","CARROT",3]]},{"farmer":["WEST"],"hands":[["PLANT","CARROT"],["COLLECT_FERTILIZER"],["EAST"],["PLANT","CARROT"],["PASS"],["PLANT","CARROT"],["PLANT","CARROT"],["PASS"],["PLANT","CARROT"],["NORTH"]],"market":[["SELL","STRAWBERRY",7],["SELL","MILK",21]]},{"farmer":["WATER"],"hands":[["WATER"],["HARVEST"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"]],"market":[["SELL","MILK",3]]},{"farmer":["PASS"],"hands":[],"market":[["SELL","FERTILIZER",6],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["PICKUP","WHEAT"],"hands":[["SOUTH"],["WATER"],["NORTH"],["PICKUP","WHEAT",4],["PICKUP","WHEAT",4],["PICKUP","FERTILIZER"],["PASS"]],"market":[["HIRE"],["HIRE"],["HIRE"],[],["BUY_PRODUCT","FERTILIZER",2],["SELL","FERTILIZER",1]]},{"farmer":["NORTH"],"hands":[["WEST"],["WEST"],["EAST"],["FEED"],["NORTH"],["WEST"],["NORTH"],["PICKUP","WHEAT"],["PICKUP","WHEAT",4],["WEST"]],"market":[["SELL","FERTILIZER",2],["SELL","FERTILIZER",1]]},{"farmer":["NORTH"],"hands":[["WEST"],["PASS"],["EAST"],["CARE"],["FEED"],["SOUTH"],["PICKUP","WHEAT",2],["NORTH"],["FEED"],["WEST"]],"market":[["SELL","FERTILIZER",2],["SELL","FERTILIZER",1]]},{"farmer":["NORTH"],"hands":[["WEST"],["WATER"],["EAST"],["COLLECT_FERTILIZER"],["CARE"],["SOUTH"],["NORTH"],["NORTH"],["PASS"],["WATER"]],"market":[["SELL","FERTILIZER",1],["SELL","WHEAT",4]]},{"farmer":["FEED"],"hands":[["WEST"],["SOUTH"],["WATER"],["HARVEST"],["COLLECT_FERTILIZER"],["WATER"],["NORTH"],["PASS"],["PASS"],["NORTH"]],"market":[["SELL","FERTILIZER",2]]},{"farmer":["CARE"],"hands":[["WEST"],["WATER"],["NORTH"],["WEST"],["WEST"],["WEST"],["FEED"],["FEED"],["CARE"],["NORTH"]],"market":[]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["NORTH"],["EAST"],["PASS"],["FEED"],["FEED"],["FERTILIZE"],["CARE"],["CARE"],["COLLECT_FERTILIZER"],["NORTH"]],"market":[["BUY_SEED","CARROT",1]]},{"farmer":["NORTH"],"hands":[["WATER"],["WATER"],["WATER"],["CARE"],["COLLECT_FERTILIZER"],["WATER"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["EAST"],["WATER"]],"market":[["BUY_SEED","CARROT",2]]},{"farmer":["WATER"],"hands":[["SOUTH"],["SOUTH"],["HARVEST"],["COLLECT_FERTILIZER"],["HARVEST"],["SOUTH"],["EAST"],["HARVEST"],["FEED"],["WEST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["PASS"],["WATER"],["PLANT","CARROT"],["WEST"],["CARE"],["WATER"],["FEED"],["EAST"],["CARE"],["WATER"]],"market":[["SELL","WHEAT",8]]},{"farmer":["PLANT","CARROT"],"hands":[["WATER"],["SOUTH"],["WATER"],["FEED"],["WEST"],["HARVEST"],["CARE"],["EAST"],["COLLECT_FERTILIZER"],["WEST"]],"market":[["BUY_SEED","CARROT",2]]},{"farmer":["WATER"],"hands":[["SOUTH"],["WATER"],["EAST"],["CARE"],["FEED"],["PLANT","CARROT"],["COLLECT_FERTILIZER"],["EAST"],["EAST"],["WATER"]],"market":[["SELL","WHEAT",7],["BUY_SEED","CARROT",1]]},{"farmer":["WEST"],"hands":[["WATER"],["SOUTH"],["WATER"],["COLLECT_FERTILIZER"],["CARE"],["WATER"],["NORTH"],["WATER"],["FEED"],["NORTH"]],"market":[["SELL","WOOL",21]]},{"farmer":["WATER"],"hands":[["HARVEST"],["WATER"],["SOUTH"],["WEST"],["COLLECT_FERTILIZER"],["EAST"],["WATER"],["HARVEST"],["CARE"],["WATER"]],"market":[[],["BUY_SEED","CARROT",1]]},{"farmer":["SOUTH"],"hands":[["PLANT","CARROT"],["WEST"],["WATER"],["FEED"],["WEST"],["WATER"],["HARVEST"],["PLANT","CARROT"],["COLLECT_FERTILIZER"],["HARVEST"]],"market":[[]]},{"farmer":["SOUTH"],"hands":[["WATER"],["WATER"],["NORTH"],["CARE"],["HARVEST"],["WEST"],["PLANT","CARROT"],["WATER"],["HARVEST"],["PLANT","CARROT"]],"market":[["BUY_SEED","CARROT",2],["SELL","WHEAT",4],["SELL","EGG",1500]]},{"farmer":["FEED"],"hands":[["SOUTH"],["HARVEST"],["NORTH"],["COLLECT_FERTILIZER"],["FEED"],["WEST"],["WATER"],["EAST"],["NORTH"],["WATER"]],"market":[["SELL","MILK",3],["SELL","MILK",3],["SELL","MILK",3],["SELL","EGG",1500]]},{"farmer":["CARE"],"hands":[["WATER"],["PLANT","CARROT"],["WATER"],["SOUTH"],["COLLECT_FERTILIZER"],["WATER"],["WEST"],["WATER"],["WATER"],["EAST"]],"market":[["SELL","MILK",3],["SELL","MILK",3],["SELL","MILK",14],[],["BUY_SEED","CARROT",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["SOUTH"],["WATER"],["NORTH"],["WATER"],["CARE"],["NORTH"],["WEST"],["HARVEST"],["WEST"],["WATER"]],"market":[[],["SELL","EGG",12]]},{"farmer":["EAST"],"hands":[["WATER"],["WEST"],["WATER"],["SOUTH"],["WEST"],["WATER"],["WEST"],["PLANT","CARROT"],["HARVEST"],["NORTH"]],"market":[[],["BUY_SEED","CARROT",1]]},{"farmer":["EAST"],"hands":[["HARVEST"],["WATER"],["HARVEST"],["WATER"],["WATER"],["HARVEST"],["WATER"],["WATER"],["FEED"],["WATER"]],"market":[[],[],[],["BUY_SEED","WHEAT",1]]},{"farmer":["NORTH"],"hands":[["PLANT","CARROT"],["WEST"],["PLANT","WHEAT"],["EAST"],["WEST"],["PLANT","CARROT"],["WEST"],["NORTH"],["CARE"],["EAST"]],"market":[["SELL","EGG",750]]},{"farmer":["WATER"],"hands":[["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["WATER"]],"market":[[]]},{"farmer":["PICKUP","WHEAT"],"hands":[],"market":[["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["FEED"],"hands":[["PICKUP","WHEAT"],["NORTH"],["NORTH"],["PICKUP","WHEAT"],["PICKUP","WHEAT"],["WEST"],["NORTH"],["PICKUP","WHEAT",2],["PICKUP","WHEAT"],["SOUTH"]],"market":[["SELL","FERTILIZER",4],["SELL","WOOL",21],[],[],["BUY_PRODUCT","FERTILIZER",2],["SELL","WHEAT",4],["SELL","FERTILIZER",1]]},{"farmer":["CARE"],"hands":[["FEED"],["PICKUP","WHEAT",5],["PICKUP","WHEAT",2],["WEST"],["EAST"],["WEST"],["PICKUP","WHEAT",2],["WEST"],["NORTH"],["HARVEST"]],"market":[["SELL","FERTILIZER",1],["SELL","FERTILIZER",1]]},{"farmer":["COLLECT_FERTILIZER"],"hands":[["CARE"],["WEST"],["NORTH"],["WEST"],["EAST"],["HARVEST"],["EAST"],["WEST"],["NORTH"],["WEST"]],"market":[["SELL","FERTILIZER",9],["SELL","FERTILIZER",1]]},{"farmer":["SOUTH"],"hands":[["COLLECT_FERTILIZER"],["HARVEST"],["FEED"],["NORTH"],["FEED"],["DIG"],["FEED"],["FEED"],["FEED"],["SOUTH"]],"market":[["SELL","FERTILIZER",1],["BUY_SEED","WHEAT",2]]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["FEED"],["CARE"],["FEED"],["CARE"],["SOUTH"],["HARVEST"],["CARE"],["CARE"],["HARVEST"]],"market":[["SELL","FERTILIZER",1],["BUY_SEED","WHEAT",1]]},{"farmer":["SOUTH"],"hands":[["WEST"],["CARE"],["COLLECT_FERTILIZER"],["CARE"],["EAST"],["HARVEST"],["WEST"],["COLLECT_FERTILIZER"],["COLLECT_FERTILIZER"],["DIG"]],"market":[["BUY_SEED","WHEAT",1]]},{"farmer":["HARVEST"],"hands":[["SOUTH"],["COLLECT_FERTILIZER"],["HARVEST"],["NORTH"],["WATER"],["DIG"],["PLACE","MILK",3],["HARVEST"],["NORTH"],["SOUTH"]],"market":[]},{"farmer":["DIG"],"hands":[["HARVEST"],["NORTH"],["EAST"],["WATER"],["HARVEST"],["PLANT","WHEAT"],["EAST"],["SOUTH"],["WATER"],["HARVEST"]],"market":[["SELL","EGG",6]]},{"farmer":["PLANT","WHEAT"],"hands":[["WEST"],["HARVEST"],["NORTH"],["WEST"],["PLANT","WHEAT"],["WATER"],["NORTH"],["PLANT","WHEAT"],["HARVEST"],["DIG"]],"market":[["BUY_SEED","WHEAT",2],["SELL","EGG",6]]},{"farmer":["WATER"],"hands":[["HARVEST"],["FEED"],["FEED"],["WATER"],["WATER"],["WEST"],["FEED"],["WATER"],["PLANT","WHEAT"],["WEST"]],"market":[["SELL","WHEAT",8],["BUY_SEED","WHEAT",1],["SELL","EGG",1500]]},{"farmer":["SOUTH"],"hands":[["SOUTH"],["CARE"],["CARE"],["WEST"],["EAST"],["HARVEST"],["CARE"],["WEST"],["WATER"],["NORTH"]],"market":[["BUY_SEED","CARROT",1],["SELL","EGG",1500]]},{"farmer":["HARVEST"],"hands":[["HARVEST"],["COLLECT_FERTILIZER"],["HARVEST"],["WATER"],["WATER"],["DIG"],["COLLECT_FERTILIZER"],["HARVEST"],["NORTH"],["HARVEST"]],"market":[["SELL","WHEAT",4]]},{"farmer":["DIG"],"hands":[["DIG"],["EAST"],["EAST"],["HARVEST"],["HARVEST"],["SOUTH"],["NORTH"],["DIG"],["WATER"],["WEST"]],"market":[["SELL","EGG",1500]]},{"farmer":["SOUTH"],"hands":[["PLANT","WHEAT"],["FEED"],["WATER"],["NORTH"],["PLANT","CARROT"],["SOUTH"],["NORTH"],["PLANT","WHEAT"],["WEST"],["WATER"]],"market":[["BUY_SEED","CARROT",1],["SELL","EGG",1500]]},{"farmer":["WATER"],"hands":[["WATER"],["CARE"],["NORTH"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WATER"],["WEST"]],"market":[["BUY_SEED","WHEAT",1],["SELL","EGG",1500]]},{"farmer":["HARVEST"],"hands":[["EAST"],["COLLECT_FERTILIZER"],["WATER"],["NORTH"],["NORTH"],["SOUTH"],["NORTH"],["NORTH"],["SOUTH"],["WATER"]],"market":[]},{"farmer":["WEST"],"hands":[["EAST"],["NORTH"],["EAST"],["WATER"],["WATER"],["WATER"],["WATER"],["FEED"],["FEED"],["NORTH"]],"market":[]},{"farmer":["PASS"],"hands":[["NORTH"],["FEED"],["WATER"],["EAST"],["WEST"],["HARVEST"],["EAST"],["CARE"],["CARE"],["WATER"]],"market":[["SELL","MILK",3]]},{"farmer":["PASS"],"hands":[["NORTH"],["CARE"],["HARVEST"],["EAST"],["WEST"],["WEST"],["HARVEST"],["NORTH"],["WEST"],["NORTH"]],"market":[["SELL","MILK",18]]},{"farmer":["PASS"],"hands":[["DROP"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["WATER"],["PASS"],["EAST"],["WATER"],["WATER"],["WATER"]],"market":[]},{"farmer":["WATER"],"hands":[["EAST"],["WEST"],["WATER"],["HARVEST"],["HARVEST"],["PASS"],["WATER"],["WEST"],["NORTH"],["HARVEST"]],"market":[["SELL","MILK",3],["SELL","STRAWBERRY",1],["SELL","STRAWBERRY",1],["SELL","STRAWBERRY",18],["SELL","MILK",9]]},{"farmer":["WEST"],"hands":[["CARE"],["FEED"],["EAST"],["SOUTH"],["PLANT","WHEAT"],["PASS"],["EAST"],["SOUTH"],["WATER"],["PLANT","CARROT"]],"market":[["SELL","FERTILIZER",1000],["SELL","MILK",2250],["SELL","CARROT",1000]]},{"farmer":["WATER"],"hands":[["COLLECT_FERTILIZER"],["CARE"],["WATER"],["WATER"],["WATER"],["HARVEST"],["HARVEST"],["WATER"],["PASS"],["WATER"]],"market":[["SELL","FERTILIZER",1000],["SELL","MILK",2250],["SELL","CARROT",1000]]},{"farmer":["PICKUP","WHEAT",6],"hands":[],"market":[["SELL","STRAWBERRY",20],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["NORTH"],"hands":[["WEST"],["WEST"],["WEST"],["WEST"],["PICKUP","WHEAT",6],["WEST"],["WEST"],["PICKUP","WHEAT",5],["NORTH"]],"market":[["SELL","MILK",6],["SELL","WHEAT",2],["SELL","FERTILIZER",11],["SELL","CARROT",2],["SELL","WOOL",3],["HIRE"],["HIRE"]]},{"farmer":["NORTH"],"hands":[["WEST"],["DIG"],["DIG"],["WEST"],["WEST"],["SOUTH"],["SOUTH"],["WEST"],["NORTH"],["NORTH"],["PICKUP","WHEAT",6]],"market":[["SELL","FERTILIZER",1]]},{"farmer":["HARVEST"],"hands":[["WEST"],["WEST"],["EAST"],["WEST"],["PASS"],["SOUTH"],["DIG"],["FEED"],["NORTH"],["NORTH"],["FEED"]],"market":[]},{"farmer":["PASS"],"hands":[["WEST"],["WATER"],["EAST"],["HARVEST"],["CARE"],["BUILD_COOP"],["WEST"],["CARE"],["NORTH"],["NORTH"],["CARE"]],"market":[]},{"farmer":["PASS"],"hands":[["WEST"],["WEST"],["EAST"],["EAST"],["COLLECT_FERTILIZER"],["WEST"],["WEST"],["COLLECT_FERTILIZER"],["WATER"],["NORTH"],["COLLECT_FERTILIZER"]],"market":[["SELL","WOOL",6]]},{"farmer":["PASS"],"hands":[["WATER"],["WATER"],["EAST"],["EAST"],["NORTH"],["DIG"],["WATER"],["WEST"],["HARVEST"],["WATER"],["NORTH"]],"market":[["SELL","EGG",1500]]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["NORTH"],["NORTH"],["NORTH"],["PASS"],["SOUTH"],["EAST"],["FEED"],["SOUTH"],["EAST"],["FEED"]],"market":[]},{"farmer":["HARVEST"],"hands":[["NORTH"],["NORTH"],["WATER"],["NORTH"],["CARE"],["WATER"],["WATER"],["COLLECT_FERTILIZER"],["SOUTH"],["WATER"],["CARE"]],"market":[["SELL","MILK",18]]},{"farmer":["SOUTH"],"hands":[["WATER"],["WATER"],["EAST"],["HARVEST"],["COLLECT_FERTILIZER"],["WEST"],["EAST"],["NORTH"],["SOUTH"],["EAST"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["HARVEST"],"hands":[["HARVEST"],["HARVEST"],["NORTH"],["EAST"],["NORTH"],["WEST"],["SOUTH"],["FEED"],["SOUTH"],["WATER"],["NORTH"]],"market":[]},{"farmer":["PLACE","MILK",8],"hands":[["NORTH"],["NORTH"],["WATER"],["NORTH"],["PASS"],["WATER"],["WATER"],["CARE"],["PLACE","CARROT",3],["HARVEST"],["FEED"]],"market":[["SELL","CARROT",3]]},{"farmer":["EAST"],"hands":[["NORTH"],["WATER"],["HARVEST"],["HARVEST"],["WEST"],["EAST"],["WEST"],["WEST"],["EAST"],["WEST"],["COLLECT_FERTILIZER"]],"market":[]},{"farmer":["EAST"],"hands":[["WATER"],["HARVEST"],["NORTH"],["SOUTH"],["WEST"],["WATER"],["SOUTH"],["SOUTH"],["EAST"],["NORTH"],["EAST"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["EAST"],"hands":[["HARVEST"],["EAST"],["WATER"],["SOUTH"],["SOUTH"],["HARVEST"],["SOUTH"],["COLLECT_FERTILIZER"],["NORTH"],["WATER"],["COLLECT_FERTILIZER"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["FEED"],"hands":[["NORTH"],["WATER"],["HARVEST"],["SOUTH"],["HARVEST"],["NORTH"],["WATER"],["CARE"],["WATER"],["HARVEST"],["CARE"]],"market":[["SELL","WHEAT",1126],["SELL","CARROT",1000]]},{"farmer":["CARE"],"hands":[["WATER"],["HARVEST"],["WEST"],["PLACE","WOOL",12],["EAST"],["WATER"],["WEST"],["PASS"],["NORTH"],["WEST"],["SOUTH"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["EAST"],"hands":[["HARVEST"],["NORTH"],["WEST"],["WEST"],["EAST"],["WEST"],["WATER"],["PASS"],["WATER"],["WEST"],["FEED"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","MILK",1125]]},{"farmer":["NORTH"],"hands":[["EAST"],["WATER"],["WEST"],["NORTH"],["SOUTH"],["WATER"],["HARVEST"],["PASS"],["EAST"],["WATER"],["CARE"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["WATER"],"hands":[["WATER"],["HARVEST"],["WEST"],["NORTH"],["PLACE","WHEAT",1000],["HARVEST"],["PASS"],["EAST"],["WATER"],["WEST"],["COLLECT_FERTILIZER"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["EAST"],"hands":[["HARVEST"],["EAST"],["SOUTH"],["SOUTH"],["PLACE","FERTILIZER",1000],["NORTH"],["PASS"],["EAST"],["NORTH"],["WATER"],["SOUTH"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["NORTH"],"hands":[["SOUTH"],["WATER"],["SOUTH"],["NORTH"],["PLACE","EGG",1000],["WATER"],["PASS"],["EAST"],["NORTH"],["HARVEST"],["FEED"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000]]},{"farmer":["NORTH"],"hands":[["WATER"],["HARVEST"],["PLACE","WHEAT",4],["SOUTH"],["PASS"],["PASS"],["PASS"],["PLACE","WHEAT"],["WATER"],["PASS"],["CARE"]],"market":[["SELL","WHEAT",750],["SELL","FERTILIZER",1000]]},{"farmer":["WATER"],"hands":[["HARVEST"],["PASS"],["DROP"],["PASS"],["PASS"],["PASS"],["PASS"],["DROP"],["HARVEST"],["PASS"],["COLLECT_FERTILIZER"]],"market":[["SELL","WHEAT",750],["SELL","FERTILIZER",1000]]},{"farmer":["WEST"],"hands":[],"market":[["SELL","WHEAT",27],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"],["HIRE"]]},{"farmer":["WEST"],"hands":[["NORTH"],["EAST"],["WEST"],["WEST"],["WEST"],["EAST"],["WEST"],["WEST"],["WEST"]],"market":[["SELL","CARROT",33],["SELL","MILK",3],["HIRE"],["HIRE"],["BUY_PRODUCT","FERTILIZER",2],["SELL","FERTILIZER",1]]},{"farmer":["SOUTH"],"hands":[["NORTH"],["NORTH"],["SOUTH"],["SOUTH"],["WEST"],["NORTH"],["WEST"],["SOUTH"],["NORTH"],["SOUTH"],["EAST"]],"market":[["SELL","FERTILIZER",11]]},{"farmer":["WEST"],"hands":[["NORTH"],["NORTH"],["WEST"],["WEST"],["HARVEST"],["NORTH"],["WEST"],["WEST"],["WEST"],["SOUTH"],["EAST"]],"market":[]},{"farmer":["WATER"],"hands":[["NORTH"],["EAST"],["WATER"],["WEST"],["NORTH"],["EAST"],["WEST"],["WATER"],["HARVEST"],["PASS"],["EAST"]],"market":[]},{"farmer":["HARVEST"],"hands":[["WATER"],["WATER"],["HARVEST"],["WEST"],["NORTH"],["EAST"],["WATER"],["HARVEST"],["EAST"],["SOUTH"],["EAST"]],"market":[]},{"farmer":["SOUTH"],"hands":[["HARVEST"],["HARVEST"],["SOUTH"],["WEST"],["EAST"],["WATER"],["HARVEST"],["NORTH"],["EAST"],["WATER"],["WATER"]],"market":[]},{"farmer":["SOUTH"],"hands":[["SOUTH"],["SOUTH"],["SOUTH"],["WATER"],["EAST"],["HARVEST"],["SOUTH"],["NORTH"],["EAST"],["HARVEST"],["HARVEST"]],"market":[]},{"farmer":["WATER"],"hands":[["EAST"],["WATER"],["WATER"],["HARVEST"],["EAST"],["NORTH"],["PASS"],["EAST"],["EAST"],["NORTH"],["PASS"]],"market":[]},{"farmer":["HARVEST"],"hands":[["WATER"],["HARVEST"],["HARVEST"],["SOUTH"],["HARVEST"],["EAST"],["SOUTH"],["EAST"],["HARVEST"],["NORTH"],["PASS"]],"market":[["SELL","WHEAT",2]]},{"farmer":["NORTH"],"hands":[["HARVEST"],["EAST"],["SOUTH"],["SOUTH"],["SOUTH"],["WATER"],["SOUTH"],["EAST"],["SOUTH"],["DROP"],["PASS"]],"market":[]},{"farmer":["NORTH"],"hands":[["EAST"],["WATER"],["EAST"],["WATER"],["SOUTH"],["HARVEST"],["HARVEST"],["EAST"],["EAST"],["NORTH"],["WEST"]],"market":[]},{"farmer":["EAST"],"hands":[["WATER"],["HARVEST"],["WATER"],["HARVEST"],["HARVEST"],["WEST"],["EAST"],["WEST"],["WEST"],["NORTH"],["WEST"]],"market":[["SELL","CARROT",4]]},{"farmer":["EAST"],"hands":[["HARVEST"],["SOUTH"],["HARVEST"],["EAST"],["EAST"],["WEST"],["EAST"],["WEST"],["WEST"],["WEST"],["WEST"]],"market":[]},{"farmer":["EAST"],"hands":[["WEST"],["WATER"],["EAST"],["EAST"],["EAST"],["SOUTH"],["EAST"],["DROP"],["DROP"],["NORTH"],["WEST"]],"market":[["SELL","FERTILIZER",1],["SELL","WHEAT",2],["SELL","WOOL",12]]},{"farmer":["DROP"],"hands":[["WEST"],["HARVEST"],["NORTH"],["NORTH"],["HARVEST"],["SOUTH"],["EAST"],["PASS"],["PASS"],["HARVEST"],["DROP"]],"market":[["SELL","WOOL",24],["SELL","MILK",21],["SELL","FERTILIZER",1],["SELL","WHEAT",2],["SELL","CARROT",3]]},{"farmer":["EAST"],"hands":[["HARVEST"],["WEST"],["NORTH"],["NORTH"],["PASS"],["SOUTH"],["NORTH"],["PASS"],["PASS"],["EAST"],["PASS"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","TOMATO",1000],["SELL","STRAWBERRY",1000],["SELL","MELON",1000],["SELL","MILK",2250],["SELL","WOOL",3000],["SELL","FERTILIZER",1000]]},{"farmer":["NORTH"],"hands":[["SOUTH"],["WEST"],["NORTH"],["NORTH"],["PASS"],["HARVEST"],["NORTH"],["PASS"],["PASS"],["SOUTH"],["PASS"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","TOMATO",1000],["SELL","STRAWBERRY",1000],["SELL","MELON",1000],["SELL","MILK",2250],["SELL","WOOL",3000],["SELL","FERTILIZER",1000]]},{"farmer":["NORTH"],"hands":[["SOUTH"],["WEST"],["NORTH"],["WATER"],["WEST"],["WEST"],["NORTH"],["PASS"],["PASS"],["SOUTH"],["PASS"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","TOMATO",1000],["SELL","STRAWBERRY",1000],["SELL","MELON",1000],["SELL","MILK",2250],["SELL","WOOL",3000],["SELL","FERTILIZER",1000]]},{"farmer":["HARVEST"],"hands":[["SOUTH"],["DROP"],["DROP"],["HARVEST"],["HARVEST"],["WEST"],["PLACE","CARROT",1000],["PASS"],["PASS"],["DROP"],["PASS"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","TOMATO",1000],["SELL","STRAWBERRY",1000],["SELL","MELON",1000],["SELL","MILK",2250],["SELL","WOOL",3000],["SELL","FERTILIZER",1000]]},{"farmer":["SOUTH"],"hands":[["DROP"],["PASS"],["PASS"],["EAST"],["WEST"],["PLACE","CARROT",1000],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","TOMATO",1000],["SELL","STRAWBERRY",1000],["SELL","MELON",1000],["SELL","MILK",2250],["SELL","WOOL",3000],["SELL","FERTILIZER",1000]]},{"farmer":["PLACE","MILK",1000],"hands":[["PASS"],["PASS"],["PASS"],["EAST"],["PLACE","MILK",1000],["PLACE","WHEAT",1000],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"]],"market":[["SELL","WHEAT",750],["SELL","CARROT",1000],["SELL","TOMATO",1000],["SELL","STRAWBERRY",1000],["SELL","MILK",2250],["SELL","WOOL",3000],["SELL","FERTILIZER",1000]]},{"farmer":["PASS"],"hands":[["PASS"],["PASS"],["PASS"],["DROP"],["PLACE","WOOL",1000],["PLACE","WOOL",1000],["PASS"],["PASS"],["PASS"],["PASS"],["PASS"]],"market":[["SELL","CARROT",1000],["SELL","WHEAT",750],["SELL","WOOL",3000]]}]')
_SETTINGS = {'hand_align': True, 'weed_repair': True, 'sell_lead': True, 'front_run': True, 'budget_guard': False, 'room_guard': True, 'clamp_sells': True, 'dead_stock': False, 'terminal_liquidation': True}
_PLAN = json.loads('') if False else None
_IMPL = make_agent({0: _TAPE}, router=None, opponent_plan=_PLAN, **_SETTINGS)

def agent(observation, configuration=None):
    try:
        return _IMPL(observation, configuration)
    except Exception:
        return {'farmer': ['PASS'], 'hands': [], 'market': []}



In [ ]:
import tarfile
import os

SRC = "main.py"
OUT = "submission.tar.gz"

assert os.path.exists(SRC), f"{SRC} not found in working directory"

with tarfile.open(OUT, "w:gz") as tar:
    tar.add(SRC, arcname="main.py")

print(f"Wrote {OUT} ({os.path.getsize(OUT)} bytes) containing:")
with tarfile.open(OUT, "r:gz") as tar:
    for member in tar.getmembers():
        print(f"  {member.name}  ({member.size} bytes)")


Apache-2.0. Credits above.
